<a href="https://colab.research.google.com/github/SamiraKheradmand/IASC-ASCE-Transformer-SHM/blob/main/notebooks/02_split_methods/chronological/Chronological_transformer_PSO_15sensor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import scipy.io as sio
from scipy.io.matlab._mio5_params import mat_struct
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

from pathlib import Path
import re
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!ls "/content/drive/My Drive/ASCE_IASC"

figures					random_window_split_seed42.csv
MAT_data				Results
processed_zero_mean			txt_all_15_sensors
PSO_results				txt_selected_sensors
random_block_assignment_45s_seed42.csv


In [ ]:
!ls "/content/drive/MyDrive/ASCE_IASC/PSO_results"

chronological_combined_pareto_seed42.csv
chronological_MOPSO_15sensor_evaluations_psoseed123.csv
chronological_MOPSO_15sensor_evaluations_psoseed2026.csv
chronological_MOPSO_15sensor_evaluations_psoseed7.csv
chronological_MOPSO_15sensor_history_psoseed123.csv
chronological_MOPSO_15sensor_history_psoseed2026.csv
chronological_MOPSO_15sensor_history_psoseed7.csv
chronological_MOPSO_15sensor_history_seed42.csv
chronological_MOPSO_15sensor_pareto_psoseed123.csv
chronological_MOPSO_15sensor_pareto_psoseed2026.csv
chronological_MOPSO_15sensor_pareto_psoseed7.csv
chronological_MOPSO_15sensor_pareto_seed42.csv
chronological_MOPSO_15sensor_state_seed42.pkl
chronological_MOPSO_pareto_seed42.csv
chronological_phase3_model_seed_cache.csv
chronological_phase3_model_seed_results.csv
chronological_PSO_15sensor_cache_seed42.csv
chronological_PSO_15sensor_history_seed42.csv
chronological_PSO_history_seed42.csv
chronological_PSO_subset_cache_seed42.csv
final_candidates_v1
importance_bottomk_validation
i

In [ ]:
from pathlib import Path

import numpy as np
import scipy.io as sio


# مسیر فایل‌ها
DATA_DIR = Path("/content/drive/MyDrive/ASCE_IASC/MAT_data")

# فقط 15 سنسور واقعی سازه
CHANNELS = [f"DA{i:02d}" for i in range(1, 16)]

# پیدا کردن فایل‌های 9 حالت سازه
mat_files = sorted(DATA_DIR.glob("shm*.mat"))

print("Folder exists:", DATA_DIR.exists())
print("Number of files:", len(mat_files))
print("Number of channels:", len(CHANNELS))


def load_ambient_file(file_path):
    """
    خواندن یک فایل Ambient از داده‌های IASC-ASCE
    """

    mat = sio.loadmat(file_path,squeeze_me=True,struct_as_record=False)

    dasy = mat["dasy"]
    dasy_dscr = mat["dasy_dscr"]
    fs = float(mat["fsdasy"])

    signals = []
    descriptions = []

    for i, channel in enumerate(CHANNELS, start=1):

        signal = np.asarray(getattr(dasy, channel)).reshape(-1)

        description_name = f"DAdscr{i:02d}"
        description = getattr(dasy_dscr, description_name)

        signals.append(signal)
        descriptions.append(description)

    # شکل خروجی:
    # (تعداد نمونه‌های زمانی، تعداد سنسورها)
    X_raw = np.column_stack(signals)

    return X_raw, fs, descriptions

Folder exists: True
Number of files: 9
Number of channels: 15


In [ ]:
all_cases = {}

for file_path in mat_files:

    case_name = file_path.stem
    case_number = int(case_name[3:5])

    X_raw, fs, descriptions = load_ambient_file(file_path)

    all_cases[case_name] = {
        "X_raw": X_raw,
        "fs": fs,
        "channels": CHANNELS.copy(),
        "descriptions": descriptions,
        "case_number": case_number,
        "label": case_number - 1,
        "file_path": file_path
    }

    print(
        f"{case_name} | "
        f"shape: {X_raw.shape} | "
        f"fs: {fs} Hz | "
        f"label: {case_number - 1}"

    )
print( f"description: {pd.DataFrame(descriptions)}")

shm01a | shape: (60000, 15) | fs: 200.0 Hz | label: 0
shm02a | shape: (60000, 15) | fs: 200.0 Hz | label: 1
shm03a | shape: (60000, 15) | fs: 200.0 Hz | label: 2
shm04a | shape: (60000, 15) | fs: 200.0 Hz | label: 3
shm05a | shape: (60000, 15) | fs: 200.0 Hz | label: 4
shm06a | shape: (45568, 15) | fs: 200.0 Hz | label: 5
shm07a | shape: (180000, 15) | fs: 200.0 Hz | label: 6
shm08a | shape: (180000, 15) | fs: 200.0 Hz | label: 7
shm09a | shape: (180000, 15) | fs: 200.0 Hz | label: 8
description:                                                0
0   Base West side - EPI sensor X direction (N+)
1      Base Center - EPI sensor Y direction (W+)
2   Base East side - EPI sensor X direction (N+)
3    1st Floor - N/S EPI Sensor at West end (N+)
4      1st Floor - E/W FBA Sensor at Center (W+)
5    1st Floor - N/S FBA Sensor at East end (N+)
6    2nd Floor - N/S FBA Sensor at West end (N+)
7      2nd Floor - E/W EPI Sensor at Center (W+)
8    2nd Floor - N/S EPI Sensor at East end (N+)
9    3rd

In [ ]:
print(all_cases["shm01a"]["X_raw"].shape)
print(all_cases["shm01a"]["channels"])


(60000, 15)
['DA01', 'DA02', 'DA03', 'DA04', 'DA05', 'DA06', 'DA07', 'DA08', 'DA09', 'DA10', 'DA11', 'DA12', 'DA13', 'DA14', 'DA15']


In [ ]:
# ============================================================
# Step 2: Chronological split and Butterworth high-pass filter
# Filter is applied separately to Train, Validation and Test
# ============================================================

import pandas as pd
from scipy.signal import butter, sosfiltfilt


# ----------------------------
# Split settings
# ----------------------------
TRAIN_RATIO = 0.60
VAL_RATIO = 0.20

GAP_SEC = 0
EDGE_SEC = 2


# ----------------------------
# High-pass filter settings
# ----------------------------
CUTOFF = 0.1
ORDER = 2


def highpass_filter(X, fs):

    """
    Apply a Butterworth high-pass filter to all sensor channels.
    X shape:(samples, sensors)
    """

    sos = butter(N=ORDER, Wn=CUTOFF,btype="highpass",fs=fs,output="sos")
    X_filtered = sosfiltfilt(sos,X,axis=0)

    return X_filtered

In [ ]:
train_data = {}
val_data = {}
test_data = {}

split_report = []

for case_name in sorted(all_cases.keys()):

    # Raw data
    X = all_cases[case_name]["X_raw"]
    fs = all_cases[case_name]["fs"]

    n_samples = len(X)

    gap_size = int(GAP_SEC * fs)
    edge_size = int(EDGE_SEC * fs)

    # ----------------------------
    # Chronological split boundaries
    # ----------------------------
    train_end = int(TRAIN_RATIO * n_samples)

    val_start = train_end + gap_size
    val_end = val_start + int(VAL_RATIO * n_samples)

    test_start = val_end + gap_size

    if test_start >= n_samples:
        raise ValueError(f"{case_name}: signal is too short for this gap.")

    # ----------------------------
    # Split raw signal
    # ----------------------------
    X_train_raw = X[:train_end]
    X_val_raw = X[val_start:val_end]
    X_test_raw = X[test_start:]

    # Check split lengths
    if min(len(X_train_raw), len(X_val_raw), len(X_test_raw)) <= 2 * edge_size:
        raise ValueError(f"{case_name}: EDGE_SEC is too large.")

    # ----------------------------
    # Filter each split separately
    # ----------------------------
    X_train_filtered = highpass_filter(X_train_raw, fs)
    X_val_filtered  = highpass_filter(X_val_raw, fs)
    X_test_filtered = highpass_filter(X_test_raw, fs)

    # ----------------------------
    # Remove filter edges
    # ----------------------------
    X_train_filtered = X_train_filtered[edge_size:-edge_size]
    X_val_filtered  = X_val_filtered[edge_size:-edge_size]
    X_test_filtered = X_test_filtered[edge_size:-edge_size]

    # ----------------------------
    # Save filtered data
    # ----------------------------
    train_data[case_name] = X_train_filtered
    val_data[case_name] = X_val_filtered
    test_data[case_name] = X_test_filtered

    split_report.append({
        "case": case_name,
        "total_samples": n_samples,

        "train_samples_filtered": len(X_train_filtered),
        "val_samples_filtered": len(X_val_filtered),
        "test_samples_filtered":len(X_test_filtered),
        "train_duration_sec":len(X_train_filtered) / fs,
        "val_duration_sec":len(X_val_filtered) / fs,
        "test_duration_sec":len(X_test_filtered) / fs
    })


split_report_df = pd.DataFrame(split_report)

display(split_report_df)

,case,total_samples,train_samples_filtered,val_samples_filtered,test_samples_filtered,train_duration_sec,val_duration_sec,test_duration_sec
0,shm01a,60000,35200,11200,11200,176.0,56.000,56.000
1,shm02a,60000,35200,11200,11200,176.0,56.000,56.000
2,shm03a,60000,35200,11200,11200,176.0,56.000,56.000
3,shm04a,60000,35200,11200,11200,176.0,56.000,56.000
4,shm05a,60000,35200,11200,11200,176.0,56.000,56.000
5,shm06a,45568,26540,8313,8315,132.7,41.565,41.575
6,shm07a,180000,107200,35200,35200,536.0,176.000,176.000
7,shm08a,180000,107200,35200,35200,536.0,176.000,176.000
8,shm09a,180000,107200,35200,35200,536.0,176.000,176.000


In [ ]:
for case_name in sorted(train_data.keys()):

    print(f"{case_name} | "
          f"Train: {train_data[case_name].shape} | "
          f"Validation: {val_data[case_name].shape} | "
          f"Test: {test_data[case_name].shape}" )

shm01a | Train: (35200, 15) | Validation: (11200, 15) | Test: (11200, 15)
shm02a | Train: (35200, 15) | Validation: (11200, 15) | Test: (11200, 15)
shm03a | Train: (35200, 15) | Validation: (11200, 15) | Test: (11200, 15)
shm04a | Train: (35200, 15) | Validation: (11200, 15) | Test: (11200, 15)
shm05a | Train: (35200, 15) | Validation: (11200, 15) | Test: (11200, 15)
shm06a | Train: (26540, 15) | Validation: (8313, 15) | Test: (8315, 15)
shm07a | Train: (107200, 15) | Validation: (35200, 15) | Test: (35200, 15)
shm08a | Train: (107200, 15) | Validation: (35200, 15) | Test: (35200, 15)
shm09a | Train: (107200, 15) | Validation: (35200, 15) | Test: (35200, 15)


In [ ]:
filter_check = []

for case_name in sorted(all_cases.keys()):

    raw_train_end = int(TRAIN_RATIO * len(all_cases[case_name]["X_raw"]))
    X_train_raw = all_cases[case_name]["X_raw"][:raw_train_end]
    X_train_filtered = train_data[case_name]

    filter_check.append({
        "case": case_name,
        "mean_abs_raw": np.mean(np.abs(np.mean(X_train_raw, axis=0))),
        "mean_abs_filtered": np.mean(np.abs(np.mean(X_train_filtered,axis=0)))
    })

filter_check_df = pd.DataFrame(filter_check)
display(filter_check_df)

,case,mean_abs_raw,mean_abs_filtered
0,shm01a,0.001260,1.125002e-06
1,shm02a,0.001564,2.848653e-07
2,shm03a,0.000955,3.674356e-07
3,shm04a,0.001096,3.553501e-07
4,shm05a,0.001285,8.238489e-07
5,shm06a,0.001239,2.023264e-06
6,shm07a,0.001139,9.591543e-08
7,shm08a,0.001083,9.232525e-08
8,shm09a,0.001077,1.732560e-07


In [ ]:
# ============================================================
# Step 3: Select sensor channels
# ============================================================

# selected_sensor_indices = [8, 11, 13, 14]  # DA09, DA12, DA14, DA15
# selected_sensor_indices = [0,3]
selected_sensor_indices = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14]

selected_sensor_names = [CHANNELS[i] for i in selected_sensor_indices]

train_selected = {}
val_selected = {}
test_selected = {}

for case_name in train_data.keys():
    train_selected[case_name] = train_data[case_name][:, selected_sensor_indices]
    val_selected[case_name]   = val_data[case_name][:, selected_sensor_indices]
    test_selected[case_name]  = test_data[case_name][:, selected_sensor_indices]

print("Selected sensors:", selected_sensor_names)

for case_name in sorted(train_selected.keys()):
    print(
        f"Case {case_name}: "
        f"train={train_selected[case_name].shape}, "
        f"val={val_selected[case_name].shape}, "
        f"test={test_selected[case_name].shape}"
    )

Selected sensors: ['DA01', 'DA02', 'DA03', 'DA04', 'DA05', 'DA06', 'DA07', 'DA08', 'DA09', 'DA10', 'DA11', 'DA12', 'DA13', 'DA14', 'DA15']
Case shm01a: train=(35200, 15), val=(11200, 15), test=(11200, 15)
Case shm02a: train=(35200, 15), val=(11200, 15), test=(11200, 15)
Case shm03a: train=(35200, 15), val=(11200, 15), test=(11200, 15)
Case shm04a: train=(35200, 15), val=(11200, 15), test=(11200, 15)
Case shm05a: train=(35200, 15), val=(11200, 15), test=(11200, 15)
Case shm06a: train=(26540, 15), val=(8313, 15), test=(8315, 15)
Case shm07a: train=(107200, 15), val=(35200, 15), test=(35200, 15)
Case shm08a: train=(107200, 15), val=(35200, 15), test=(35200, 15)
Case shm09a: train=(107200, 15), val=(35200, 15), test=(35200, 15)


In [ ]:
# ============================================================
# Step 4: Global train-based normalization
# One mean and standard deviation for each selected sensor
# ============================================================

EPS = 1e-8
train_concat = np.concatenate(
    [train_selected[case_name] for case_name in sorted(train_selected.keys())],axis=0)


print("Combined Train shape:", train_concat.shape)
train_mean = train_concat.mean(axis=0,keepdims=True,dtype=np.float64)
train_std = train_concat.std(axis=0,keepdims=True,dtype=np.float64)


# Prevent division by zero
train_std_safe = np.where(train_std < EPS,1.0,train_std)


print("Train mean shape:", train_mean.shape)
print("Train std shape:", train_std_safe.shape)

Combined Train shape: (524140, 15)
Train mean shape: (1, 15)
Train std shape: (1, 15)


In [ ]:
train_norm = {}
val_norm = {}
test_norm = {}


for case_name in sorted(train_selected.keys()):

    train_norm[case_name] = ((train_selected[case_name] - train_mean)/ train_std_safe).astype(np.float32)
    val_norm[case_name]   = ((val_selected[case_name] - train_mean)/ train_std_safe).astype(np.float32)
    test_norm[case_name]  = ((test_selected[case_name] - train_mean)/ train_std_safe).astype(np.float32)


print("Normalization completed.")

Normalization completed.


In [ ]:
train_norm_concat = np.concatenate(
    [ train_norm[case_name] for case_name in sorted(train_norm.keys())],axis=0)

normalized_train_mean = train_norm_concat.mean(axis=0)
normalized_train_std = train_norm_concat.std(axis=0)

print("Train mean after normalization:")
print(np.round(normalized_train_mean,5))

print("\nTrain std after normalization:")
print(np.round(normalized_train_std,5))

Train mean after normalization:
[ 0.  0. -0. -0.  0.  0. -0. -0.  0. -0.  0.  0.  0. -0.  0.]

Train std after normalization:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [ ]:
# ============================================================
# Step 5: Create non-overlapping time windows
# ============================================================

WINDOW_SIZE = 512
STEP_SIZE = 512

FS_DASY = 200

print("Window size:", WINDOW_SIZE, "samples")
print("Window duration:", WINDOW_SIZE / FS_DASY, "seconds")
print("Step size:", STEP_SIZE, "samples")
print("Step duration:", STEP_SIZE / FS_DASY, "seconds")

Window size: 512 samples
Window duration: 2.56 seconds
Step size: 512 samples
Step duration: 2.56 seconds


In [ ]:
def make_windows(split_data):

    all_windows = []
    all_labels = []
    meta_rows = []

    for case_name in sorted(split_data.keys()):

        X = split_data[case_name]

        n_samples = len(X)

        # shm01a -> scenario 1 -> label 0
        scenario_number = int(case_name[3:5])
        label = scenario_number - 1

        window_number = 0

        for start in range(0, n_samples - WINDOW_SIZE + 1,STEP_SIZE):

            end = start + WINDOW_SIZE

            window = X[start:end, :]

            all_windows.append(window)
            all_labels.append(label)

            meta_rows.append({
                "case": case_name,
                "scenario": scenario_number,
                "label": label,
                "window": window_number,
                "start_sample": start,
                "end_sample": end
            })

            window_number += 1

    if len(all_windows) == 0:
        raise ValueError("No windows were created.")

    X_windows = np.stack(all_windows,axis=0).astype(np.float32)

    y_labels = np.array(all_labels,dtype=np.int64)

    meta_df = pd.DataFrame(meta_rows)

    return X_windows, y_labels, meta_df

In [ ]:
X_train, y_train, train_meta_df = make_windows(train_norm)

X_val, y_val, val_meta_df = make_windows(val_norm)

X_test, y_test, test_meta_df = make_windows(test_norm)

In [ ]:
print("Window shapes:")
print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

print("\nLabel shapes:")
print("y_train:", y_train.shape)
print("y_val:  ", y_val.shape)
print("y_test: ", y_test.shape)

Window shapes:
X_train: (1018, 512, 15)
X_val:   (325, 512, 15)
X_test:  (325, 512, 15)

Label shapes:
y_train: (1018,)
y_val:   (325,)
y_test:  (325,)


In [ ]:
print("\nTrain class counts:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nValidation class counts:")
print(pd.Series(y_val).value_counts().sort_index())

print("\nTest class counts:")
print(pd.Series(y_test).value_counts().sort_index())


Train class counts:
0     68
1     68
2     68
3     68
4     68
5     51
6    209
7    209
8    209
Name: count, dtype: int64

Validation class counts:
0    21
1    21
2    21
3    21
4    21
5    16
6    68
7    68
8    68
Name: count, dtype: int64

Test class counts:
0    21
1    21
2    21
3    21
4    21
5    16
6    68
7    68
8    68
Name: count, dtype: int64


In [ ]:
import numpy as np
import torch

from torch.utils.data import TensorDataset, DataLoader


# ----------------------------
# بررسی داده‌ها قبل از Tensor
# ----------------------------
print("X_train shape:", X_train.shape)
print("X_val shape:  ", X_val.shape)
print("X_test shape: ", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_val shape:  ", y_val.shape)
print("y_test shape: ", y_test.shape)


# تعداد داده و برچسب باید برابر باشد
assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)

# بررسی NaN و Inf
assert np.isfinite(X_train).all(), "X_train contains NaN or Inf."
assert np.isfinite(X_val).all(), "X_val contains NaN or Inf."
assert np.isfinite(X_test).all(), "X_test contains NaN or Inf."

print("Data check completed.")

X_train shape: (1018, 512, 15)
X_val shape:   (325, 512, 15)
X_test shape:  (325, 512, 15)
y_train shape: (1018,)
y_val shape:   (325,)
y_test shape:  (325,)
Data check completed.


In [ ]:
# ============================================================
# Step 6: Convert arrays to PyTorch tensors
# ============================================================

import random
import numpy as np
import torch

from torch.utils.data import TensorDataset, DataLoader


# ----------------------------
# Reproducibility
# ----------------------------
SPLIT_SEED = None
MODEL_SEED = 42

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Reproducibility for CUDA
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Warn if a nondeterministic operation is used
    torch.use_deterministic_algorithms(True,warn_only=True)

set_seed(MODEL_SEED)

# Inputs must be float32
X_train_tensor = torch.from_numpy(X_train.astype(np.float32, copy=False))
X_val_tensor = torch.from_numpy(X_val.astype(np.float32, copy=False))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32, copy=False))


# Labels for CrossEntropyLoss must be int64
y_train_tensor = torch.from_numpy(y_train.astype(np.int64, copy=False))
y_val_tensor = torch.from_numpy(y_val.astype(np.int64, copy=False))
y_test_tensor = torch.from_numpy(y_test.astype(np.int64, copy=False))


print("\nTensor shapes:")
print("X_train_tensor:", X_train_tensor.shape)
print("y_train_tensor:", y_train_tensor.shape)

print("X_val_tensor:", X_val_tensor.shape)
print("y_val_tensor:", y_val_tensor.shape)

print("X_test_tensor:", X_test_tensor.shape)
print("y_test_tensor:", y_test_tensor.shape)


Tensor shapes:
X_train_tensor: torch.Size([1018, 512, 15])
y_train_tensor: torch.Size([1018])
X_val_tensor: torch.Size([325, 512, 15])
y_val_tensor: torch.Size([325])
X_test_tensor: torch.Size([325, 512, 15])
y_test_tensor: torch.Size([325])


In [ ]:
# ============================================================
# Create Dataset and DataLoader
# ============================================================

train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
val_dataset   = TensorDataset(X_val_tensor,y_val_tensor)
test_dataset  = TensorDataset(X_test_tensor,y_test_tensor)

BATCH_SIZE = 64

# برای تکرارپذیری Shuffle داده‌های Train
loader_generator = torch.Generator()
loader_generator.manual_seed(MODEL_SEED)


train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,generator=loader_generator)
val_loader   = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False)
test_loader  = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False)

In [ ]:
print("Number of train samples:", len(train_dataset))
print("Number of val samples:  ", len(val_dataset))
print("Number of test samples: ", len(test_dataset))

print("\nNumber of train batches:", len(train_loader))
print("Number of val batches:  ", len(val_loader))
print("Number of test batches: ", len(test_loader))


X_batch, y_batch = next(iter(train_loader))

print("\nOne train batch:")
print("X_batch shape:", X_batch.shape)
print("y_batch shape:", y_batch.shape)
print("First 10 labels:", y_batch[:10])

Number of train samples: 1018
Number of val samples:   325
Number of test samples:  325

Number of train batches: 16
Number of val batches:   6
Number of test batches:  6

One train batch:
X_batch shape: torch.Size([64, 512, 15])
y_batch shape: torch.Size([64])
First 10 labels: tensor([4, 8, 8, 6, 6, 7, 2, 1, 8, 8])


In [ ]:
# ============================================================
# Step 7: Define raw time-series Transformer
# Input shape: (batch, time, sensors)
# ============================================================

import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score

# ----------------------------
# Device
# ----------------------------
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)


# ----------------------------
# Learnable positional encoding
# ----------------------------
class LearnablePositionalEncoding(nn.Module):

    def __init__(self, seq_len, d_model):
        super().__init__()
        self.position = nn.Parameter(torch.zeros(1, seq_len, d_model))
        nn.init.trunc_normal_(self.position,std=0.02)

    def forward(self, x):
        seq_len = x.shape[1]
        return x + self.position[:, :seq_len, :]

Device: cuda


In [ ]:
# ============================================================
# Step 7: Define raw time-series Transformer
# Input shape: (batch, time, sensors)
# ============================================================

import torch.nn as nn

# --- Added these definitions to ensure the model parameters are defined ---
NUM_CLASSES = 9
SEQ_LEN = WINDOW_SIZE
INPUT_DIM = X_train_tensor.shape[2]
D_MODEL = 64
NUM_HEADS = 8
NUM_LAYERS = 2
DIM_FEEDFORWARD = 64
DROPOUT = 0.1
# --- End of added definitions ---

class TransformerTimeSeriesClassifier(nn.Module):
    def __init__(
        self,
        input_dim,
        seq_len,
        num_classes,
        d_model=64,
        num_heads=8,
        num_layers=2,
        dim_feedforward=64,
        dropout=0.10
    ):
        super().__init__()

        self.seq_len = seq_len
        self.d_model = d_model

        self.input_projection = nn.Linear(input_dim, d_model)
        self.pos_embedding = nn.Parameter(torch.zeros(1, seq_len, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="relu",
            batch_first=True,
            norm_first=True
        )

        self.transformer_encoder = nn.TransformerEncoder(encoder_layer,num_layers=num_layers)
        self.final_norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model,num_classes)
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.pos_embedding, mean=0.0, std=0.02)

    def forward(self, x):
        # x shape: (batch, time, channels)

        x = self.input_projection(x)
        x = x + self.pos_embedding[:, :x.size(1), :]
        x = self.transformer_encoder(x)
        x = self.final_norm(x)
        # Mean pooling over time
        x = x.mean(dim=1)
        logits = self.classifier(x)
        return logits



In [ ]:
set_seed(MODEL_SEED)
model = TransformerTimeSeriesClassifier(
    input_dim=INPUT_DIM,
    seq_len=SEQ_LEN,
    num_classes=NUM_CLASSES,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT
).to(device)

print(model)

# ----------------------------
# Count trainable parameters
# ----------------------------
def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\nTrainable parameters:", count_trainable_parameters(model))


# ----------------------------
# Check one forward pass
# ----------------------------
for xb, yb in train_loader:
    xb = xb.to(device)

    logits = model(xb)

    print("\nInput batch shape:", xb.shape)
    print("Output logits shape:", logits.shape)

    break

TransformerTimeSeriesClassifier(
  (input_projection): Linear(in_features=15, out_features=64, bias=True)
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=64, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=64, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (final_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (classifier): Linear(in_features=64, out_features=9, bias=True)
)

Trainable parameters: 84937

Input batch shape:

/tmp/ipykernel_1577/3525986821.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer,num_layers=num_layers)


In [ ]:
# ============================================================
# Step 8: Loss function and optimizer
# ============================================================

class_counts = torch.bincount(y_train_tensor, minlength=NUM_CLASSES).float()
if torch.any(class_counts == 0):
    raise ValueError("At least one class has no training samples.")

class_weights = (len(y_train_tensor) / (NUM_CLASSES * class_counts))
class_weights = class_weights.to(device)

print("Class counts and weights:")

for class_id in range(NUM_CLASSES):

    print(
        f"Scenario {class_id + 1} | "
        f"count: {int(class_counts[class_id])} | "
        f"weight: {class_weights[class_id].item():.4f}"
    )

Class counts and weights:
Scenario 1 | count: 68 | weight: 1.6634
Scenario 2 | count: 68 | weight: 1.6634
Scenario 3 | count: 68 | weight: 1.6634
Scenario 4 | count: 68 | weight: 1.6634
Scenario 5 | count: 68 | weight: 1.6634
Scenario 6 | count: 51 | weight: 2.2179
Scenario 7 | count: 209 | weight: 0.5412
Scenario 8 | count: 209 | weight: 0.5412
Scenario 9 | count: 209 | weight: 0.5412


In [ ]:
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)

print("Loss function:", criterion)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", LEARNING_RATE)

Loss function: CrossEntropyLoss()
Optimizer: AdamW
Learning rate: 0.0002


In [ ]:
# Reset early stopping
best_val_macro_f1 = -1.0
best_val_loss = float("inf")

best_epoch = 0
best_model_state = None

patience_counter = 0
print("New experiment started")
print("Model_Seed:", MODEL_SEED)
print("Heads:", NUM_HEADS)
print("Dropout:", DROPOUT)

New experiment started
Model_Seed: 42
Heads: 8
Dropout: 0.1


In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import copy
import torch

NUM_EPOCHS = 100
PATIENCE = 20
GRAD_CLIP = 1.0
MIN_DELTA = 1e-4

LABELS_ORDER = np.arange(NUM_CLASSES)

transformer_history = {
    "epoch": [],
    "train_loss": [],
    "train_acc": [],
    "train_macro_f1": [],
    "val_loss": [],
    "val_acc": [],
    "val_macro_f1": [],
}

best_val_macro_f1 = -1.0
best_epoch = 0
best_model_state = None
epochs_without_improvement = 0


def run_one_epoch_transformer(model, data_loader, criterion, optimizer=None, device="cpu"):
    """
    If optimizer is given: training mode
    If optimizer is None: evaluation mode
    """

    is_training = optimizer is not None

    if is_training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for xb, yb in data_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        if is_training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_training):
            logits = model(xb)
            loss = criterion(logits, yb)

            if is_training:
                loss.backward()

                # Gradient clipping helps Transformer training stability
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)

                optimizer.step()

        total_loss += loss.item() * xb.size(0)
        preds = torch.argmax(logits, dim=1)
        all_preds.append(preds.detach().cpu())
        all_labels.append(yb.detach().cpu())

    avg_loss = total_loss / len(data_loader.dataset)
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    return avg_loss, acc, macro_f1




In [ ]:
# # Standalone baseline training
# # Keep this cell, but do not run it during candidate evaluation


# for epoch in range(1, NUM_EPOCHS + 1):

#     train_loss, train_acc, train_macro_f1 = run_one_epoch_transformer(
#         model=model,
#         data_loader=train_loader,
#         criterion=criterion,
#         optimizer=optimizer,
#         device=device
#     )

#     val_loss, val_acc, val_macro_f1 = run_one_epoch_transformer(
#         model=model,
#         data_loader=val_loader,
#         criterion=criterion,
#         optimizer=None,
#         device=device
#     )

#     transformer_history["epoch"].append(epoch)
#     transformer_history["train_loss"].append(train_loss)
#     transformer_history["train_acc"].append(train_acc)
#     transformer_history["train_macro_f1"].append(train_macro_f1)
#     transformer_history["val_loss"].append(val_loss)
#     transformer_history["val_acc"].append(val_acc)
#     transformer_history["val_macro_f1"].append(val_macro_f1)

#     # Save best model based on validation Macro-F1
#     if val_macro_f1 > best_val_macro_f1 + MIN_DELTA:
#         best_val_macro_f1 = val_macro_f1
#         best_epoch = epoch # Assign epoch here
#         best_model_state = copy.deepcopy(model.state_dict())
#         epochs_without_improvement = 0
#     else:
#         epochs_without_improvement += 1

#     if epoch == 1 or epoch % 5 == 0:
#         print(
#             f"Epoch {epoch:03d} | "
#             f"Train Loss: {train_loss:.4f} | "
#             f"Train Acc: {train_acc:.4f} | "
#             f"Train Macro-F1: {train_macro_f1:.4f} | "
#             f"Val Loss: {val_loss:.4f} | "
#             f"Val Acc: {val_acc:.4f} | "
#             f"Val Macro-F1: {val_macro_f1:.4f}"
#         )

#     # Early stopping
#     if epochs_without_improvement >= PATIENCE:
#         print(f"\nEarly stopping at epoch {epoch}.")
#         break

In [ ]:
# if best_model_state is None:
#     raise RuntimeError("No best model was saved.")

# model.load_state_dict(best_model_state)

# print("\nTraining finished.")
# print("Best epoch:",best_epoch)
# print("Best validation Macro-F1:",best_val_macro_f1)

In [ ]:
# ============================================================
# Final evaluation control
# ============================================================

# Keep False during development/tuning.
# Set True only for final test evaluation.
RUN_TEST = False

In [ ]:
# ============================================================
# Final evaluation
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

import pandas as pd
import torch


train_eval_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


def predict_with_model_transformer(
    model,
    data_loader,
    device="cpu"
):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for xb, yb in data_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb)
            preds = torch.argmax(logits,dim=1)
            all_preds.append(preds.cpu())
            all_labels.append(yb.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    return all_labels, all_preds


def final_evaluate_transformer(
    model,
    data_loader,
    split_name,
    device="cpu"
):

    y_true, y_pred = predict_with_model_transformer(
        model=model,
        data_loader=data_loader,
        device=device
    )

    acc = accuracy_score(
        y_true,
        y_pred
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    print(f"\n===== {split_name} =====")
    print(f"Accuracy:    {acc:.4f}")
    print(f"Macro-F1:    {macro_f1:.4f}")
    print(f"Weighted-F1: {weighted_f1:.4f}")

    return {
        "split": split_name,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "y_true": y_true,
        "y_pred": y_pred
    }

In [ ]:
# model.eval()

# transformer_train_results = final_evaluate_transformer(
#     model,
#     train_eval_loader,
#     "TRAIN",
#     device=device
# )

# transformer_val_results = final_evaluate_transformer(
#     model,
#     val_loader,
#     "VALIDATION",
#     device=device
# )

In [ ]:
# transformer_test_results = None

# if RUN_TEST:

#     transformer_test_results = final_evaluate_transformer(
#         model,
#         test_loader,
#         "TEST",
#         device=device
#     )

# else:

#     print("\nTest evaluation skipped.")

In [ ]:
# summary_rows = [
#     {
#         "model": "Raw Transformer",
#         "preprocessing":
#             "Butterworth high-pass + Train-based normalization",
#         "selected_sensors":
#             len(selected_sensor_names),
#         "split":
#             transformer_train_results["split"],
#         "accuracy":
#             transformer_train_results["accuracy"],
#         "macro_f1":
#             transformer_train_results["macro_f1"],
#         "weighted_f1":
#             transformer_train_results["weighted_f1"]
#     },

#     {
#         "model": "Raw Transformer",
#         "preprocessing":
#             "Butterworth high-pass + Train-based normalization",
#         "selected_sensors":
#             len(selected_sensor_names),
#         "split":
#             transformer_val_results["split"],
#         "accuracy":
#             transformer_val_results["accuracy"],
#         "macro_f1":
#             transformer_val_results["macro_f1"],
#         "weighted_f1":
#             transformer_val_results["weighted_f1"]
#     }
# ]

In [ ]:
# if RUN_TEST:

#     summary_rows.append(
#         {
#             "model": "Raw Transformer",
#             "preprocessing":
#                 "Butterworth high-pass + Train-based normalization",
#             "selected_sensors":
#                 len(selected_sensor_names),
#             "split":
#                 transformer_test_results["split"],
#             "accuracy":
#                 transformer_test_results["accuracy"],
#             "macro_f1":
#                 transformer_test_results["macro_f1"],
#             "weighted_f1":
#                 transformer_test_results["weighted_f1"]
#         }
#     )

In [ ]:
# transformer_summary_results = pd.DataFrame(summary_rows)
# display(transformer_summary_results)

In [ ]:
# # ============================================================
# # Step 11: Test confusion matrix
# # ============================================================


# if RUN_TEST:

#     y_true_test = (
#         transformer_test_results["y_true"]
#     )

#     y_pred_test = (
#         transformer_test_results["y_pred"]
#     )

#     cm_transformer_test = confusion_matrix(
#         y_true_test,
#         y_pred_test,
#         labels=np.arange(NUM_CLASSES)
#     )

#     disp = ConfusionMatrixDisplay(
#         confusion_matrix=cm_transformer_test,
#         display_labels=[
#             f"S{i+1}"
#             for i in range(NUM_CLASSES)
#         ]
#     )

#     disp.plot(
#         values_format="d",
#         cmap="Blues",
#         colorbar=False
#     )

#     plt.title(
#         "Test Confusion Matrix"
#     )

#     plt.grid(False)
#     plt.show()

In [ ]:
# Chronological + High-pass + Raw Transformer + 12 sensors

# Test Accuracy    = 0.9892 ± 0.0018
# Test Macro-F1    = 0.9930 ± 0.0021
# Test Weighted-F1 = 0.9893 ± 0.0018

In [ ]:
# # ============================================================
# # Step 12: Training curves
# # ============================================================

# plt.figure(figsize=(8, 5))

# plt.plot(
#     transformer_history["epoch"],
#     transformer_history["train_loss"],
#     label="Train Loss"
# )

# plt.plot(
#     transformer_history["epoch"],
#     transformer_history["val_loss"],
#     label="Validation Loss"
# )

# plt.axvline(
#     best_epoch,
#     linestyle="--",
#     label=f"Best epoch = {best_epoch}"
# )

# plt.xlabel("Epoch")
# plt.ylabel("Loss")
# plt.title("Training and Validation Loss")
# plt.legend()
# plt.grid(True)

# plt.tight_layout()
# plt.show()

In [ ]:
# plt.figure(figsize=(8, 5))

# plt.plot(
#     transformer_history["epoch"],
#     transformer_history["train_macro_f1"],
#     label="Train Macro-F1"
# )

# plt.plot(
#     transformer_history["epoch"],
#     transformer_history["val_macro_f1"],
#     label="Validation Macro-F1"
# )

# plt.axvline(
#     best_epoch,
#     linestyle="--",
#     label=f"Best epoch = {best_epoch}"
# )

# plt.xlabel("Epoch")
# plt.ylabel("Macro-F1")
# plt.title("Training and Validation Macro-F1")
# plt.legend()
# plt.grid(True)

# plt.tight_layout()
# plt.show()

In [ ]:
# ============================================================
# PSO Phase 1
# Basic settings and cache
# ============================================================

import time

PSO_RESULTS_DIR = Path("/content/drive/MyDrive/ASCE_IASC/PSO_results")

PSO_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CACHE_PATH = (
    PSO_RESULTS_DIR /
    "chronological_PSO_15sensor_cache_seed42.csv"
)

# Checks
assert RUN_TEST is False
assert len(selected_sensor_names) == 15
assert X_train_tensor.shape[2] == 15

print("Sensors:", selected_sensor_names)
print("Number of sensors:", len(selected_sensor_names))
print("Model seed:", MODEL_SEED)
print("Cache path:", CACHE_PATH)
print("Test closed:", RUN_TEST is False)

Sensors: ['DA01', 'DA02', 'DA03', 'DA04', 'DA05', 'DA06', 'DA07', 'DA08', 'DA09', 'DA10', 'DA11', 'DA12', 'DA13', 'DA14', 'DA15']
Number of sensors: 15
Model seed: 42
Cache path: /content/drive/MyDrive/ASCE_IASC/PSO_results/chronological_PSO_15sensor_cache_seed42.csv
Test closed: True


In [ ]:
# ============================================================
# PSO Phase 1
# Convert binary mask to sensor subset
# ============================================================
#  ساخت یک بردار باینری متشکل از 0 و 1  نان دهنده وضعیت سنسورها ، وجود یا هدم وجود سنسور
def get_sensor_subset(mask):

    mask = np.asarray(mask, dtype=int)

    if len(mask) != len(selected_sensor_names):
        raise ValueError("Mask length is not correct.")

    if not np.all(np.isin(mask, [0, 1])):
        raise ValueError("Mask must contain only 0 and 1.")

    if mask.sum() == 0:
        raise ValueError("At least one sensor must be selected.")

    sensor_indices = np.where(mask == 1)[0].tolist()

    sensor_names = [selected_sensor_names[i] for i in sensor_indices]

    mask_key = "".join(str(x) for x in mask)

    return sensor_indices, sensor_names, mask_key

In [ ]:
# ============================================================
# PSO Phase 1
# Create DataLoaders for one sensor subset
# ============================================================

def make_subset_loaders(mask):

    sensor_indices, sensor_names, mask_key = (get_sensor_subset(mask))

    X_train_sub = X_train_tensor[:, :, sensor_indices]
    X_val_sub = X_val_tensor[:, :, sensor_indices]

    train_dataset_sub = TensorDataset( X_train_sub,y_train_tensor )
    val_dataset_sub = TensorDataset(X_val_sub,y_val_tensor )

    # Fresh shuffle with the same model seed
    generator = torch.Generator()
    generator.manual_seed(MODEL_SEED)

    train_loader_sub = DataLoader(train_dataset_sub,batch_size=BATCH_SIZE,shuffle=True,generator=generator)
    val_loader_sub = DataLoader(val_dataset_sub,batch_size=BATCH_SIZE,shuffle=False)

    return train_loader_sub, val_loader_sub

In [ ]:
# ============================================================
# PSO Phase 1
# Cache functions
# ============================================================

def load_pso_cache():

    if not CACHE_PATH.exists():
        return pd.DataFrame()

    cache_df = pd.read_csv(CACHE_PATH, dtype={"mask_key": str})

    cache_df["mask_key"] = (cache_df["mask_key"].str.zfill(len(selected_sensor_names)))

    return cache_df


def save_pso_result(result):

    cache_df = load_pso_cache()
    cache_df = pd.concat([cache_df, pd.DataFrame([result])],ignore_index=True)
    cache_df = cache_df.drop_duplicates(subset=["mask_key", "model_seed"],keep="last")

    temp_path = CACHE_PATH.with_suffix(".tmp")

    cache_df.to_csv(temp_path, index=False)
    temp_path.replace(CACHE_PATH)

In [ ]:
# ============================================================
# Evaluate one subset
# Optional saving of the best model weights
# ============================================================

def evaluate_sensor_subset(
    mask,
    use_cache=True,
    checkpoint_dir=None
):

    assert RUN_TEST is False, "Keep RUN_TEST=False."

    indices, sensors, key = get_sensor_subset(mask)

    # --------------------------------------------------------
    # 1. Read an existing result
    # --------------------------------------------------------

    if use_cache:

        cache_df = load_pso_cache()

        if not cache_df.empty:

            found = cache_df[
                (cache_df["mask_key"] == key)
                & (cache_df["model_seed"] == MODEL_SEED)
            ]

            if not found.empty:

                result = found.iloc[-1].to_dict()

                if result["sensors"] != "|".join(sensors):
                    raise ValueError(
                        "Cached sensor names do not match the mask."
                    )

                print(
                    "Cached:", sensors,
                    "| Seed:", MODEL_SEED,
                    "| Val F1:", result["val_macro_f1"]
                )

                return result

    # --------------------------------------------------------
    # 2. Fresh model and fresh DataLoaders
    # --------------------------------------------------------

    print("\nTraining:", sensors, "| Seed:", MODEL_SEED)

    set_seed(MODEL_SEED)

    train_sub, val_sub = make_subset_loaders(mask)

    model_config = {
        "input_dim": len(sensors),
        "seq_len": SEQ_LEN,
        "num_classes": NUM_CLASSES,
        "d_model": D_MODEL,
        "num_heads": NUM_HEADS,
        "num_layers": NUM_LAYERS,
        "dim_feedforward": DIM_FEEDFORWARD,
        "dropout": DROPOUT
    }

    model_sub = TransformerTimeSeriesClassifier(**model_config).to(device)

    optimizer_sub = torch.optim.AdamW(
        model_sub.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    best_f1 = -1.0
    best_epoch_sub = 0

    best_acc = 0.0
    best_loss = float("inf")
    best_weights = None

    waiting = 0
    start = time.time()

    # --------------------------------------------------------
    # 3. Train using the existing epoch function
    # --------------------------------------------------------

    for epoch in range(1, NUM_EPOCHS + 1):

        _, _, train_f1 = run_one_epoch_transformer(
            model_sub,
            train_sub,
            criterion,
            optimizer_sub,
            device
        )

        val_loss, val_acc, val_f1 = run_one_epoch_transformer(
            model_sub,
            val_sub,
            criterion,
            optimizer=None,
            device=device
        )

        if not np.isfinite(
            [val_loss, val_acc, val_f1]
        ).all():
            raise RuntimeError(
                "Non-finite validation result. Nothing cached."
            )

        if val_f1 > best_f1 + MIN_DELTA:

            best_f1 = val_f1
            best_acc = val_acc
            best_loss = val_loss
            best_epoch_sub = epoch

            waiting = 0

            if checkpoint_dir is not None:

                best_weights = {
                    name: value.detach().cpu().clone()
                    for name, value
                    in model_sub.state_dict().items()
                }

        else:
            waiting += 1

        if epoch == 1 or epoch % 5 == 0:

            print(
                f"Epoch {epoch:03d} | "
                f"Train F1: {train_f1:.4f} | "
                f"Val F1: {val_f1:.4f} | "
                f"Best: {best_f1:.4f}"
            )

        if waiting >= PATIENCE:
            print("Early stopping at epoch", epoch)
            break

    if best_epoch_sub == 0:
        raise RuntimeError(
            "No valid training epoch was completed."
        )

    # --------------------------------------------------------
    # 4. Prepare the result
    # --------------------------------------------------------

    result = {
        "mask_key": key,
        "sensors": "|".join(sensors),
        "n_sensors": len(sensors),
        "model_seed": int(MODEL_SEED),

        "val_macro_f1": float(best_f1),
        "val_accuracy": float(best_acc),
        "val_loss": float(best_loss),

        "best_epoch": best_epoch_sub,
        "epochs_run": epoch,
        "training_seconds": time.time() - start,

        "weights_path": "",
        "source_file": "new_training"
    }

    # --------------------------------------------------------
    # 5. Save best weights, not last-epoch weights
    # --------------------------------------------------------

    if checkpoint_dir is not None:

        checkpoint_dir = Path(checkpoint_dir)
        checkpoint_dir.mkdir(parents=True, exist_ok=True)

        weights_path = (
            checkpoint_dir
            / f"{key}_seed{MODEL_SEED}.pt"
        )

        temp_path = weights_path.with_suffix(".tmp")

        torch.save(
            {
                "model_state_dict": best_weights,
                "model_config": model_config,

                "sensors": sensors,
                "model_seed": int(MODEL_SEED),
                "best_epoch": best_epoch_sub,
                "val_macro_f1": float(best_f1),

                "train_mean": torch.from_numpy(
                    train_mean[:, indices].copy()
                ),

                "train_std": torch.from_numpy(
                    train_std_safe[:, indices].copy()
                ),

                "preprocessing": {
                    "fs": FS_DASY,
                    "cutoff": CUTOFF,
                    "order": ORDER,
                    "edge_sec": EDGE_SEC,
                    "gap_sec": GAP_SEC,
                    "window_size": WINDOW_SIZE,
                    "step_size": STEP_SIZE
                }
            },
            temp_path
        )

        temp_path.replace(weights_path)
        result["weights_path"] = str(weights_path)

    save_pso_result(result)

    print(
        "Finished | Best epoch:", best_epoch_sub,
        "| Best Val F1:", round(best_f1, 6)
    )

    del model_sub, optimizer_sub, best_weights

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

In [ ]:
# ============================================================
# Check saved PSO results
# ============================================================

cache_df = load_pso_cache()

print("Number of cached results:", len(cache_df))

if len(cache_df) > 0:

    print(
        cache_df[
            [
                "mask_key",
                "sensors",
                "n_sensors",
                "model_seed",
                "val_macro_f1",
                "best_epoch",
                "training_seconds"
            ]
        ].tail()
    )

Number of cached results: 383
            mask_key                                       sensors  n_sensors  \
378  111001011001101  DA01|DA02|DA03|DA06|DA08|DA09|DA12|DA13|DA15          9   
379  100010011001011            DA01|DA05|DA08|DA09|DA12|DA14|DA15          7   
380  001101001101110       DA03|DA04|DA06|DA09|DA10|DA12|DA13|DA14          8   
381  000111011100010            DA04|DA05|DA06|DA08|DA09|DA10|DA14          7   
382  001010111110000            DA03|DA05|DA07|DA08|DA09|DA10|DA11          7   

     model_seed  val_macro_f1  best_epoch  training_seconds  
378          42      1.000000          28         81.674127  
379          42      0.983899          73        157.838739  
380          42      1.000000          30         84.793142  
381          42      0.994706          20         67.777459  
382          42      0.996479          31         86.493657  


In [ ]:
# ============================================================
# PSO Phase 1
# Binary PSO settings
# ============================================================

N_PARTICLES = 8
N_ITERATIONS = 5

W_MAX = 0.9
W_MIN = 0.4

C1 = 1.5
C2 = 1.5

LAMBDA = 0.05

PSO_SEED = 2026

print("Particles:", N_PARTICLES)
print("Iterations:", N_ITERATIONS)
print("Lambda:", LAMBDA)
print("PSO seed:", PSO_SEED)

Particles: 8
Iterations: 5
Lambda: 0.05
PSO seed: 2026


In [ ]:
# ============================================================
# PSO Phase 1
# Fitness function
# ============================================================

def calculate_fitness(mask):

    result = evaluate_sensor_subset( mask, use_cache=True )

    val_f1 = result["val_macro_f1"]
    n_sensors = result["n_sensors"]

    fitness = ( val_f1 - LAMBDA * (n_sensors /  len(selected_sensor_names)))

    return fitness, val_f1, n_sensors

In [ ]:
# ============================================================
# PSO Phase 1
# Initialize particles
# ============================================================

rng = np.random.default_rng(PSO_SEED)

n_sensors = len(selected_sensor_names)

positions = np.zeros((N_PARTICLES, n_sensors), dtype=int)

# Start with different numbers of sensors
initial_sizes = [2, 3, 4, 5, 6, 8, 10, 15]

for i, k in enumerate(initial_sizes):

    selected = rng.choice(n_sensors, size=k, replace=False)

    positions[i, selected] = 1

# Initial velocities
velocities = rng.uniform(-1, 1, size=(N_PARTICLES, n_sensors))
print("Initial population:")

for i in range(N_PARTICLES):

    _, sensors, _ = get_sensor_subset(positions[i])

    print( f"Particle {i + 1}: "
        f"{len(sensors)} sensors -> {sensors}" )

Initial population:
Particle 1: 2 sensors -> ['DA03', 'DA12']
Particle 2: 3 sensors -> ['DA06', 'DA08', 'DA09']
Particle 3: 4 sensors -> ['DA05', 'DA08', 'DA12', 'DA15']
Particle 4: 5 sensors -> ['DA02', 'DA05', 'DA09', 'DA11', 'DA14']
Particle 5: 6 sensors -> ['DA02', 'DA03', 'DA07', 'DA08', 'DA10', 'DA15']
Particle 6: 8 sensors -> ['DA02', 'DA03', 'DA08', 'DA10', 'DA11', 'DA12', 'DA13', 'DA15']
Particle 7: 10 sensors -> ['DA02', 'DA03', 'DA04', 'DA05', 'DA06', 'DA07', 'DA09', 'DA10', 'DA13', 'DA15']
Particle 8: 15 sensors -> ['DA01', 'DA02', 'DA03', 'DA04', 'DA05', 'DA06', 'DA07', 'DA08', 'DA09', 'DA10', 'DA11', 'DA12', 'DA13', 'DA14', 'DA15']


In [ ]:
# # ============================================================
# # PSO Phase 1
# # Run Binary PSO
# # ============================================================

# # Personal best
# pbest_positions = positions.copy()
# pbest_fitness = np.zeros(N_PARTICLES)

# # Evaluate initial population
# print("Evaluating initial population...\n")

# for i in range(N_PARTICLES):

#     fitness, val_f1, k = calculate_fitness( positions[i] )
#     pbest_fitness[i] = fitness

#     print(
#         f"Particle {i + 1} | "
#         f"Sensors: {k} | "
#         f"Val F1: {val_f1:.4f} | "
#         f"Fitness: {fitness:.4f}"
#     )


# # Global best
# best_index = np.argmax(pbest_fitness)
# gbest_position = pbest_positions[best_index].copy()
# gbest_fitness = pbest_fitness[best_index]

# # Save PSO progress
# history = []

# best_so_far = gbest_fitness
# no_improvement = 0

# # ------------------------------------------------------------
# # PSO iterations
# # ------------------------------------------------------------

# for iteration in range(N_ITERATIONS):

#     print(
#         f"\n========== Iteration {iteration + 1} "
#         f"/ {N_ITERATIONS} =========="
#     )

#     # Decrease inertia from W_MAX to W_MIN
#     if N_ITERATIONS > 1:
#         w = ( W_MAX - (W_MAX - W_MIN) * iteration / (N_ITERATIONS - 1))

#     else:
#         w = W_MIN


#     for i in range(N_PARTICLES):

#         r1 = rng.random(n_sensors)
#         r2 = rng.random(n_sensors)

#         # Update velocity
#         velocities[i] = (
#             w * velocities[i]
#             + C1 * r1 * (pbest_positions[i] - positions[i])
#             + C2 * r2 * (gbest_position - positions[i]))

#         # Keep velocity in a reasonable range
#         velocities[i] = np.clip(velocities[i],-4, 4 )

#         # Binary PSO sigmoid
#         probability = (1 / (1 + np.exp(-velocities[i])))
#         positions[i] = (rng.random(n_sensors) < probability).astype(int)

#         # Do not allow an empty subset
#         if positions[i].sum() == 0:

#             random_sensor = rng.integers(n_sensors )
#             positions[i, random_sensor] = 1


#         # Evaluate new position
#         fitness, val_f1, k = calculate_fitness(positions[i])


#         # Update personal best
#         if fitness > pbest_fitness[i]:

#             pbest_fitness[i] = fitness
#             pbest_positions[i] = (positions[i].copy())


#     # Update global best
#     best_index = np.argmax(pbest_fitness)

#     new_best_fitness = pbest_fitness[best_index]

#     if new_best_fitness > gbest_fitness:

#         gbest_fitness = new_best_fitness
#         gbest_position = pbest_positions[best_index].copy()


#     # Current best subset
#     _, best_sensors, _ = get_sensor_subset(gbest_position)
#     best_result = evaluate_sensor_subset(gbest_position,use_cache=True)

#     history.append({
#         "iteration": iteration + 1,
#         "fitness": gbest_fitness,
#         "val_macro_f1":
#             best_result["val_macro_f1"],
#         "n_sensors":
#             best_result["n_sensors"],
#         "sensors":
#             "|".join(best_sensors)
#     })


#     print("\nBest so far:")
#     print("Sensors:", best_sensors)
#     print("Validation Macro-F1:",round(best_result["val_macro_f1"], 4))
#     print("Fitness:", round(gbest_fitness, 4))


#     # Early stopping
#     if gbest_fitness > best_so_far + 1e-6:

#         best_so_far = gbest_fitness
#         no_improvement = 0

#     else:
#         no_improvement += 1


#     if no_improvement >= 3:

#         print(
#             "\nPSO stopped: "
#             "no improvement for 3 iterations." )

#         break

In [ ]:
# # ============================================================
# # PSO Phase 1
# # Save and show results
# # ============================================================

# history_df = pd.DataFrame(history)

# HISTORY_PATH = (
#     PSO_RESULTS_DIR /
#     "chronological_PSO_15sensor_history_seed42.csv")

# history_df.to_csv(HISTORY_PATH,index=False)


# _, best_sensors, _ = get_sensor_subset(gbest_position)

# best_result = evaluate_sensor_subset(gbest_position,use_cache=True)


# print("\n==============================")
# print("PSO Phase 1 result - 15 sensors")
# print("==============================")

# print("Best sensors:")
# print(best_sensors)

# print(
#     "\nNumber of sensors:",
#     best_result["n_sensors"]
# )

# print(
#     "Validation Macro-F1:",
#     best_result["val_macro_f1"]
# )

# print(
#     "Fitness:",
#     gbest_fitness
# )

# print(
#     "\nHistory saved to:",
#     HISTORY_PATH
# )

In [ ]:
# ============================================================
# PSO Phase 1
# Summary of evaluated subsets
# ============================================================

cache_df = load_pso_cache()

# One row for each unique subset
cache_df = cache_df.drop_duplicates(
    subset=["mask_key", "model_seed"],
    keep="last"
).copy()

# Fitness used in Phase 1
cache_df["fitness"] = (
    cache_df["val_macro_f1"]
    - LAMBDA * (cache_df["n_sensors"] / len(selected_sensor_names))
)

print("Number of unique evaluated subsets:", len(cache_df))

print("\nTop 48 by fitness -->  ||| F1 - landa * (n /15)|||  :")

print(
    cache_df[
        [
            "sensors",
            "n_sensors",
            "val_macro_f1",
            "fitness"
        ]
    ]
    .sort_values("fitness", ascending=False)
    .head(48)
    .to_string(index=False)
)

Number of unique evaluated subsets: 383

Top 48 by fitness -->  ||| F1 - landa * (n /15)|||  :
                           sensors  n_sensors  val_macro_f1  fitness
          DA01|DA06|DA09|DA12|DA15          5      0.998366 0.981699
          DA05|DA06|DA07|DA08|DA13          5      0.998366 0.981699
          DA01|DA05|DA06|DA13|DA15          5      0.998366 0.981699
          DA06|DA08|DA09|DA12|DA13          5      0.996720 0.980053
     DA01|DA03|DA04|DA07|DA12|DA15          6      1.000000 0.980000
     DA01|DA04|DA09|DA11|DA12|DA15          6      1.000000 0.980000
     DA03|DA04|DA06|DA09|DA12|DA14          6      1.000000 0.980000
     DA01|DA04|DA06|DA07|DA08|DA12          6      1.000000 0.980000
     DA03|DA05|DA07|DA09|DA10|DA11          6      1.000000 0.980000
     DA01|DA03|DA05|DA09|DA13|DA15          6      1.000000 0.980000
     DA01|DA03|DA07|DA08|DA09|DA12          6      1.000000 0.980000
               DA01|DA09|DA12|DA13          4      0.993234 0.979900
        

In [ ]:
# ============================================================
# PSO Phase 2
# Pareto dominance
# ============================================================

def dominates(f1_a, k_a, f1_b, k_b):

    better_or_equal_f1 = f1_a >= f1_b
    fewer_or_equal_sensors = k_a <= k_b

    strictly_better = ( f1_a > f1_b or k_a < k_b )

    return (
        better_or_equal_f1
        and fewer_or_equal_sensors
        and strictly_better
    )

In [ ]:
# ============================================================
# PSO Phase 2
# Build Pareto archive
# ============================================================

def get_pareto_archive():

    cache_df = load_pso_cache()

    cache_df = cache_df.drop_duplicates(
        subset=["mask_key", "model_seed"],
        keep="last"
    ).copy()

    pareto_rows = []

    for i, row_i in cache_df.iterrows():

        is_dominated = False

        for j, row_j in cache_df.iterrows():

            if i == j:
                continue

            if dominates(
                row_j["val_macro_f1"],
                row_j["n_sensors"],
                row_i["val_macro_f1"],
                row_i["n_sensors"]
            ):
                is_dominated = True
                break

        if not is_dominated:
            pareto_rows.append(row_i)

    pareto_df = pd.DataFrame(pareto_rows)

    return pareto_df.sort_values(
        ["n_sensors", "val_macro_f1"],
        ascending=[True, False]
    ).reset_index(drop=True)

In [ ]:
# ============================================================
# PSO Phase 2
# Initialize MOPSO
# ============================================================

MOPSO_SEED = 2027
rng_mo = np.random.default_rng(MOPSO_SEED)

n_sensors = len(selected_sensor_names)

pareto_df = get_pareto_archive()
cache_df = load_pso_cache()

cache_df = cache_df.drop_duplicates(
    subset=["mask_key", "model_seed"],
    keep="last"
).copy()


initial_masks = []


# Start from Pareto solutions
for key in pareto_df["mask_key"]:

    mask = np.array([int(x) for x in str(key).zfill(n_sensors)])

    initial_masks.append(mask)


# Add full sensor model if it is not already included
full_mask = np.ones(n_sensors,dtype=int)

if not any(
    np.array_equal(full_mask, mask)
    for mask in initial_masks
):
    initial_masks.append(full_mask)


# If we still have fewer than N_PARTICLES,
# add already evaluated subsets from the cache
for key in cache_df["mask_key"]:

    if len(initial_masks) >= N_PARTICLES:
        break

    mask = np.array(
        [int(x) for x in str(key).zfill(n_sensors)])

    if not any(
        np.array_equal(mask, old_mask)
        for old_mask in initial_masks
    ):
        initial_masks.append(mask)


# Use only N_PARTICLES
positions_mo = np.array(initial_masks[:N_PARTICLES])

velocities_mo = rng_mo.uniform(-1,1,size=positions_mo.shape)


print("Initial MOPSO population:")

for i, mask in enumerate(positions_mo):

    _, sensors, _ = get_sensor_subset(mask)

    print(
        f"Particle {i + 1}: "
        f"{len(sensors)} sensors -> {sensors}" )

Initial MOPSO population:
Particle 1: 1 sensors -> ['DA11']
Particle 2: 2 sensors -> ['DA01', 'DA03']
Particle 3: 3 sensors -> ['DA08', 'DA09', 'DA13']
Particle 4: 4 sensors -> ['DA01', 'DA09', 'DA12', 'DA13']
Particle 5: 5 sensors -> ['DA01', 'DA06', 'DA09', 'DA12', 'DA15']
Particle 6: 5 sensors -> ['DA05', 'DA06', 'DA07', 'DA08', 'DA13']
Particle 7: 5 sensors -> ['DA01', 'DA05', 'DA06', 'DA13', 'DA15']
Particle 8: 6 sensors -> ['DA01', 'DA03', 'DA05', 'DA09', 'DA13', 'DA15']


In [ ]:
# ============================================================
# PSO Phase 2
# MOPSO checkpoint path
# ============================================================

import pickle

MOPSO_STATE_PATH = (
    PSO_RESULTS_DIR /
    "chronological_MOPSO_15sensor_state_seed42.pkl"
)

print("Checkpoint path:")
print(MOPSO_STATE_PATH)

Checkpoint path:
/content/drive/MyDrive/ASCE_IASC/PSO_results/chronological_MOPSO_15sensor_state_seed42.pkl


In [ ]:
# # ============================================================
# # PSO Phase 2
# # Run MOPSO
# # ============================================================

# # Personal best starts from initial positions
# pbest_positions_mo = positions_mo.copy()

# pbest_f1 = np.zeros(N_PARTICLES)
# pbest_k = np.zeros(N_PARTICLES, dtype=int)


# # Initial evaluation
# # These should already be available in cache
# for i in range(N_PARTICLES):

#     result = evaluate_sensor_subset(
#         positions_mo[i],
#         use_cache=True
#     )

#     pbest_f1[i] = result["val_macro_f1"]
#     pbest_k[i] = result["n_sensors"]


# history_mo = []


# for iteration in range(N_ITERATIONS):

#     print(
#         f"\n========== MOPSO Iteration "
#         f"{iteration + 1}/{N_ITERATIONS} =========="
#     )

#     # Current Pareto archive
#     pareto_df = get_pareto_archive()


#     # Decreasing inertia
#     if N_ITERATIONS > 1:

#         w = (W_MAX - (W_MAX - W_MIN) * iteration / (N_ITERATIONS - 1))

#     else:
#         w = W_MIN


#     for i in range(N_PARTICLES):

#         # --------------------------------------------
#         # Choose a leader from Pareto archive
#         # --------------------------------------------

#         available_k = pareto_df["n_sensors" ].unique()

#         leader_k = rng_mo.choice(available_k)

#         possible_leaders = pareto_df[pareto_df["n_sensors"] == leader_k]

#         leader_row = possible_leaders.sample(
#             n=1, random_state=int(rng_mo.integers(1_000_000))).iloc[0]


#         leader_position = np.array(
#             [
#                 int(x)
#                 for x in str(
#                     leader_row["mask_key"]
#                 ).zfill(n_sensors)
#             ]
#         )


#         # --------------------------------------------
#         # Binary PSO update
#         # --------------------------------------------

#         r1 = rng_mo.random(n_sensors)
#         r2 = rng_mo.random(n_sensors)


#         velocities_mo[i] = (
#             w * velocities_mo[i]
#             + C1 * r1 * ( pbest_positions_mo[i] - positions_mo[i] )
#             + C2 * r2 * ( leader_position - positions_mo[i] ))


#         velocities_mo[i] = np.clip(velocities_mo[i], -4, 4 )

#         probability = (1 / (1 + np.exp(-velocities_mo[i])))

#         new_position = (rng_mo.random(n_sensors) < probability ).astype(int)


#         # At least one sensor
#         if new_position.sum() == 0:

#             random_sensor = rng_mo.integers( n_sensors)
#             new_position[random_sensor] = 1

#         positions_mo[i] = new_position


#         # --------------------------------------------
#         # Evaluate new subset
#         # --------------------------------------------

#         result = evaluate_sensor_subset( new_position, use_cache=True )

#         new_f1 = result["val_macro_f1"]
#         new_k = result["n_sensors"]


#         # --------------------------------------------
#         # Update personal best
#         # --------------------------------------------

#         new_dominates_old = dominates(
#             new_f1,
#             new_k,
#             pbest_f1[i],
#             pbest_k[i]
#         )

#         old_dominates_new = dominates(
#             pbest_f1[i],
#             pbest_k[i],
#             new_f1,
#             new_k
#         )


#         if new_dominates_old:

#             pbest_positions_mo[i] = (new_position.copy())

#             pbest_f1[i] = new_f1
#             pbest_k[i] = new_k


#         elif not old_dominates_new:

#             # Neither solution dominates the other
#             if rng_mo.random() < 0.5:

#                 pbest_positions_mo[i] = (new_position.copy())

#                 pbest_f1[i] = new_f1
#                 pbest_k[i] = new_k


#     # Pareto front after this iteration
#     pareto_df = get_pareto_archive()


#     history_mo.append({
#         "iteration": iteration + 1,

#         "evaluated_subsets":
#             len(
#                 load_pso_cache().drop_duplicates(
#                     subset=[
#                         "mask_key",
#                         "model_seed"
#                     ]
#                 )
#             ),

#         "pareto_size":
#             len(pareto_df)
#     })
#     # --------------------------------------------------------
#     # Save MOPSO state after each iteration
#     # --------------------------------------------------------

#     state = {
#         "next_iteration": iteration + 1,
#         "positions_mo": positions_mo,
#         "velocities_mo": velocities_mo,
#         "pbest_positions_mo": pbest_positions_mo,
#         "pbest_f1": pbest_f1,
#         "pbest_k": pbest_k,
#         "history_mo": history_mo,
#         "rng_state": rng_mo.bit_generator.state
#     }

#     with open(MOPSO_STATE_PATH, "wb") as f:
#         pickle.dump(state, f)

#     print(
#         "Checkpoint saved after iteration",
#         iteration + 1
#     )


#     print("\nCurrent Pareto front:")

#     print(
#         pareto_df[
#             [
#                 "n_sensors",
#                 "val_macro_f1",
#                 "sensors"
#             ]
#         ].to_string(index=False)
#     )

In [ ]:
# ============================================================
# PSO Phase 2
# Final Pareto front
# ============================================================

pareto_df = get_pareto_archive()

print("\nFinal Pareto front:")

print(
    pareto_df[
        [
            "n_sensors",
            "val_macro_f1",
            "sensors"
        ]
    ].to_string(index=False))


PARETO_PATH = (
    PSO_RESULTS_DIR /
    "chronological_MOPSO_15sensor_pareto_seed42.csv"
)

pareto_df.to_csv(
    PARETO_PATH,
    index=False
)


n_evaluated = len(
    load_pso_cache().drop_duplicates(
        subset=["mask_key", "model_seed"]
    )
)


print(
    "\nNumber of evaluated subsets:",
    n_evaluated
)

print(
    "Pareto results saved to:",
    PARETO_PATH
)


Final Pareto front:
 n_sensors  val_macro_f1                       sensors
         1      0.332370                          DA11
         2      0.902507                     DA01|DA03
         3      0.970477                DA08|DA09|DA13
         4      0.993234           DA01|DA09|DA12|DA13
         5      0.998366      DA01|DA06|DA09|DA12|DA15
         5      0.998366      DA05|DA06|DA07|DA08|DA13
         5      0.998366      DA01|DA05|DA06|DA13|DA15
         6      1.000000 DA01|DA03|DA05|DA09|DA13|DA15
         6      1.000000 DA01|DA03|DA04|DA07|DA12|DA15
         6      1.000000 DA01|DA04|DA06|DA07|DA08|DA12
         6      1.000000 DA03|DA05|DA07|DA09|DA10|DA11
         6      1.000000 DA01|DA03|DA07|DA08|DA09|DA12
         6      1.000000 DA01|DA04|DA09|DA11|DA12|DA15
         6      1.000000 DA03|DA04|DA06|DA09|DA12|DA14

Number of evaluated subsets: 383
Pareto results saved to: /content/drive/MyDrive/ASCE_IASC/PSO_results/chronological_MOPSO_15sensor_pareto_seed42.csv


In [ ]:
# # Save MOPSO history

# history_mo_df = pd.DataFrame(history_mo)

# MOPSO_HISTORY_PATH = (PSO_RESULTS_DIR /"chronological_MOPSO_15sensor_history_seed42.csv")

# history_mo_df.to_csv(MOPSO_HISTORY_PATH, index=False)

# print("MOPSO history saved to:", MOPSO_HISTORY_PATH)

In [ ]:
# ============================================================
# Combined Pareto
# Load 12-sensor and 15-sensor results
# ============================================================

PARETO_12_PATH = (
    PSO_RESULTS_DIR /
    "chronological_MOPSO_pareto_seed42.csv"
)

PARETO_15_PATH = (
    PSO_RESULTS_DIR /
    "chronological_MOPSO_15sensor_pareto_seed42.csv"
)

pareto_12 = pd.read_csv(PARETO_12_PATH)
pareto_15 = pd.read_csv(PARETO_15_PATH)

pareto_12["search_space"] = "12 floor sensors"
pareto_15["search_space"] = "15 all sensors"

combined_df = pd.concat(
    [pareto_12, pareto_15],
    ignore_index=True
)

# Remove exact repeated solutions
combined_df = combined_df.drop_duplicates(
    subset=[
        "sensors",
        "n_sensors",
        "val_macro_f1"
    ]
).reset_index(drop=True)

print("12-sensor Pareto solutions:", len(pareto_12))
print("15-sensor Pareto solutions:", len(pareto_15))
print("Combined solutions:", len(combined_df))

12-sensor Pareto solutions: 7
15-sensor Pareto solutions: 14
Combined solutions: 21


In [ ]:
# ============================================================
# Combined Pareto
# Find non-dominated solutions
# ============================================================

combined_pareto_rows = []

for i, row_i in combined_df.iterrows():

    is_dominated = False

    for j, row_j in combined_df.iterrows():

        if i == j:
            continue

        if dominates(
            row_j["val_macro_f1"],
            row_j["n_sensors"],
            row_i["val_macro_f1"],
            row_i["n_sensors"]
        ):
            is_dominated = True
            break

    if not is_dominated:
        combined_pareto_rows.append(row_i)


combined_pareto = pd.DataFrame(
    combined_pareto_rows
)

combined_pareto = combined_pareto.sort_values(
    ["n_sensors", "val_macro_f1"],
    ascending=[True, False]
).reset_index(drop=True)

In [ ]:
# ============================================================
# Combined Pareto
# Identify Base sensors
# ============================================================

BASE_SENSORS = ["DA01", "DA02", "DA03"]


def find_base_sensors(sensor_string):

    sensors = sensor_string.split("|")

    base_used = [
        sensor
        for sensor in sensors
        if sensor in BASE_SENSORS
    ]

    return "|".join(base_used)


combined_pareto["base_sensors"] = (
    combined_pareto["sensors"]
    .apply(find_base_sensors)
)

combined_pareto["uses_base"] = (
    combined_pareto["base_sensors"] != ""
)


print("\nCombined Pareto front:")

print(
    combined_pareto[
        [
            "n_sensors",
            "val_macro_f1",
            "sensors",
            "uses_base",
            "base_sensors",
            "search_space"
        ]
    ].to_string(index=False)
)


Combined Pareto front:
 n_sensors  val_macro_f1                       sensors  uses_base base_sensors     search_space
         1      0.332370                          DA11      False                15 all sensors
         2      0.902507                     DA01|DA03       True    DA01|DA03   15 all sensors
         3      0.970477                DA08|DA09|DA13      False                15 all sensors
         4      0.993234           DA01|DA09|DA12|DA13       True         DA01   15 all sensors
         5      0.998366      DA08|DA09|DA11|DA12|DA13      False              12 floor sensors
         5      0.998366      DA01|DA06|DA09|DA12|DA15       True         DA01   15 all sensors
         5      0.998366      DA05|DA06|DA07|DA08|DA13      False                15 all sensors
         5      0.998366      DA01|DA05|DA06|DA13|DA15       True         DA01   15 all sensors
         6      1.000000 DA01|DA03|DA05|DA09|DA13|DA15       True    DA01|DA03   15 all sensors
         6      

In [ ]:
# ============================================================
# Combined Pareto
# Save results
# ============================================================

COMBINED_PARETO_PATH = (
    PSO_RESULTS_DIR /
    "chronological_combined_pareto_seed42.csv"
)

combined_pareto.to_csv(
    COMBINED_PARETO_PATH,
    index=False
)

print(
    "Combined Pareto saved to:",
    COMBINED_PARETO_PATH
)

Combined Pareto saved to: /content/drive/MyDrive/ASCE_IASC/PSO_results/chronological_combined_pareto_seed42.csv


In [ ]:
# # ============================================================
# # Base-only test
# # DA01, DA02, DA03
# # ============================================================

# base_mask = np.array([
#     0, 0, 0,   # DA01, DA02, DA03
#     0, 0, 0,   # DA04, DA05, DA06
#     0, 0, 0,   # DA07, DA08, DA09
#     0, 0, 0,   # DA10, DA11, DA12
#     1, 0, 1    # DA13, DA14, DA15
# ])

# base_result = evaluate_sensor_subset(
#     base_mask,
#     use_cache=True
# )

# print("\n==============================")
# print("Base-only result")
# print("==============================")

# print("Sensors:", base_result["sensors"])
# print("Number of sensors:", base_result["n_sensors"])
# print(
#     "Validation Macro-F1:",
#     base_result["val_macro_f1"]
# )
# print(
#     "Best epoch:",
#     base_result["best_epoch"]
# )
# print(
#     "Epochs run:",
#     base_result["epochs_run"]
# )
# print(
#     "Training time:",
#     round(
#         base_result["training_seconds"],
#         1
#     ),
#     "seconds"
# )

In [ ]:
# # ============================================================
# # Base sensor ablation
# # ============================================================

# ablation_masks = {

#     "Remove DA01": np.array([
#         0, 0, 1, 0, 1,
#         0, 0, 0, 1, 0,
#         0, 0, 1, 0, 1
#     ]),

#     "Remove DA03": np.array([
#         1, 0, 0, 0, 1,
#         0, 0, 0, 1, 0,
#         0, 0, 1, 0, 1
#     ]),

#     "Remove DA01 and DA03": np.array([
#         0, 0, 0, 0, 1,
#         0, 0, 0, 1, 0,
#         0, 0, 1, 0, 1
#     ])
# }


# ablation_results = []

# for name, mask in ablation_masks.items():

#     print("\n==============================")
#     print(name)
#     print("==============================")

#     result = evaluate_sensor_subset(
#         mask,
#         use_cache=True
#     )

#     ablation_results.append({
#         "experiment": name,
#         "sensors": result["sensors"],
#         "n_sensors": result["n_sensors"],
#         "val_macro_f1": result["val_macro_f1"],
#         "best_epoch": result["best_epoch"]
#     })

In [ ]:
# ============================================================
# Show ablation results
# ============================================================

# ablation_df = pd.DataFrame(ablation_results)

# print(
#     ablation_df.to_string(index=False)
# )

In [ ]:
# ============================================================
# Phase 3A
# Model-seed robustness
# ============================================================

MODEL_SEEDS = [42, 7, 123, 2026]

phase3_candidates = {
    "Base_2": ["DA01", "DA03"],

    "Floor_2": ["DA13", "DA15"],

    "Floor_4": ["DA05", "DA09", "DA13", "DA15"],

    "WithBase_6": ["DA01", "DA03","DA05", "DA09", "DA13", "DA15"]}


def sensor_names_to_mask(sensor_names):

    mask = np.zeros(
        len(selected_sensor_names),
        dtype=int
    )

    for sensor in sensor_names:

        index = selected_sensor_names.index(sensor)
        mask[index] = 1

    return mask


PHASE3_CACHE_PATH = (
    PSO_RESULTS_DIR /
    "chronological_phase3_model_seed_cache.csv"
)

print("Model seeds:", MODEL_SEEDS)

for name, sensors in phase3_candidates.items():
    print(name, "->", sensors)

print("Phase 3 cache:")
print(PHASE3_CACHE_PATH)

Model seeds: [42, 7, 123, 2026]
Base_2 -> ['DA01', 'DA03']
Floor_2 -> ['DA13', 'DA15']
Floor_4 -> ['DA05', 'DA09', 'DA13', 'DA15']
WithBase_6 -> ['DA01', 'DA03', 'DA05', 'DA09', 'DA13', 'DA15']
Phase 3 cache:
/content/drive/MyDrive/ASCE_IASC/PSO_results/chronological_phase3_model_seed_cache.csv


In [ ]:
# ============================================================
# Phase 3A
# Copy existing Seed=42 results
# ============================================================

PSO_CACHE_PATH = CACHE_PATH

pso_cache = load_pso_cache()

pso_cache = pso_cache.drop_duplicates(
    subset=["mask_key", "model_seed"],
    keep="last"
)


candidate_keys = []

for sensors in phase3_candidates.values():

    mask = sensor_names_to_mask(sensors)

    _, _, key = get_sensor_subset(mask)

    candidate_keys.append(key)


seed42_results = pso_cache[
    (pso_cache["model_seed"] == 42)
    &
    (pso_cache["mask_key"].isin(candidate_keys))
].copy()


# If Phase 3 cache already exists, keep its results
if PHASE3_CACHE_PATH.exists():

    phase3_cache = pd.read_csv(
        PHASE3_CACHE_PATH,
        dtype={"mask_key": str}
    )

    phase3_cache = pd.concat(
        [phase3_cache, seed42_results],
        ignore_index=True
    )

else:

    phase3_cache = seed42_results


phase3_cache = phase3_cache.drop_duplicates(
    subset=["mask_key", "model_seed"],
    keep="last"
)

phase3_cache.to_csv(
    PHASE3_CACHE_PATH,
    index=False
)


print(
    "Seed=42 results copied:",
    len(seed42_results)
)

print(
    "Results currently in Phase 3 cache:",
    len(phase3_cache)
)

Seed=42 results copied: 4
Results currently in Phase 3 cache: 16


In [ ]:
# ============================================================
# Phase 3A
# Evaluate candidates across model seeds
# ============================================================

# Use separate cache for this experiment
CACHE_PATH = PHASE3_CACHE_PATH

phase3_results = []


for candidate_name, sensors in phase3_candidates.items():

    mask = sensor_names_to_mask(sensors)

    print("\n======================================")
    print("Candidate:", candidate_name)
    print("Sensors:", sensors)
    print("======================================")

    for seed in MODEL_SEEDS:

        print("\nModel seed:", seed)

        MODEL_SEED = seed

        result = evaluate_sensor_subset(
            mask,
            use_cache=True
        )

        phase3_results.append({
            "candidate": candidate_name,
            "sensors": result["sensors"],
            "n_sensors": result["n_sensors"],
            "model_seed": seed,
            "val_macro_f1": result["val_macro_f1"],
            "best_epoch": result["best_epoch"]
        })


# Return notebook settings to the original PSO setup
MODEL_SEED = 42
CACHE_PATH = PSO_CACHE_PATH


Candidate: Base_2
Sensors: ['DA01', 'DA03']

Model seed: 42
Cached: ['DA01', 'DA03'] | Seed: 42 | Val F1: 0.9025065390317804

Model seed: 7
Cached: ['DA01', 'DA03'] | Seed: 7 | Val F1: 0.8633206145739772

Model seed: 123
Cached: ['DA01', 'DA03'] | Seed: 123 | Val F1: 0.8915804975215723

Model seed: 2026
Cached: ['DA01', 'DA03'] | Seed: 2026 | Val F1: 0.838544992157797

Candidate: Floor_2
Sensors: ['DA13', 'DA15']

Model seed: 42
Cached: ['DA13', 'DA15'] | Seed: 42 | Val F1: 0.8060017833403756

Model seed: 7
Cached: ['DA13', 'DA15'] | Seed: 7 | Val F1: 0.7979414583902548

Model seed: 123
Cached: ['DA13', 'DA15'] | Seed: 123 | Val F1: 0.7792557132603221

Model seed: 2026
Cached: ['DA13', 'DA15'] | Seed: 2026 | Val F1: 0.8041191070199933

Candidate: Floor_4
Sensors: ['DA05', 'DA09', 'DA13', 'DA15']

Model seed: 42
Cached: ['DA05', 'DA09', 'DA13', 'DA15'] | Seed: 42 | Val F1: 0.9930719182959636

Model seed: 7
Cached: ['DA05', 'DA09', 'DA13', 'DA15'] | Seed: 7 | Val F1: 0.9832674666423636


In [ ]:
# # ============================================================
# # Phase 3A
# # Results by model seed
# # ============================================================

# phase3_df = pd.DataFrame(
#     phase3_results
# )

# PHASE3_RESULTS_PATH = (
#     PSO_RESULTS_DIR /
#     "chronological_phase3_model_seed_results.csv"
# )

# phase3_df.to_csv(
#     PHASE3_RESULTS_PATH,
#     index=False
# )


# print(
#     phase3_df[
#         [
#             "candidate",
#             "n_sensors",
#             "model_seed",
#             "val_macro_f1"
#         ]
#     ].to_string(index=False)
# )

In [ ]:
# # ============================================================
# # Phase 3A
# # Mean and standard deviation
# # ============================================================

# phase3_summary = (
#     phase3_df
#     .groupby(
#         [
#             "candidate",
#             "sensors",
#             "n_sensors"
#         ]
#     )["val_macro_f1"]
#     .agg(
#         ["mean", "std", "min", "max"]
#     )
#     .reset_index()
# )


# phase3_summary = phase3_summary.sort_values(
#     "mean",
#     ascending=False
# )


# print(
#     phase3_summary.to_string(
#         index=False
#     )
# )

In [ ]:
# # ============================================================
# # MOPSO robustness
# # Settings
# # ============================================================

# MODEL_SEED = 42

# CACHE_PATH = (
#     PSO_RESULTS_DIR /
#     "chronological_PSO_15sensor_cache_seed42.csv"
# )

# MOPSO_SEEDS = [7,123,2026]

# n_sensors = len(selected_sensor_names)

# assert n_sensors == 15

# print("Model seed:", MODEL_SEED)
# print("MOPSO seeds:", MOPSO_SEEDS)
# print("Number of sensors:", n_sensors)

In [ ]:
# ============================================================
# MOPSO robustness
# Pareto front for one PSO run
# ============================================================

def get_run_pareto(run_df):

    run_df = run_df.drop_duplicates(
        subset=["mask_key"],
        keep="last"
    ).copy()

    pareto_rows = []

    for i, row_i in run_df.iterrows():

        is_dominated = False

        for j, row_j in run_df.iterrows():

            if i == j:
                continue

            if dominates(
                row_j["val_macro_f1"],
                row_j["n_sensors"],
                row_i["val_macro_f1"],
                row_i["n_sensors"]
            ):
                is_dominated = True
                break

        if not is_dominated:
            pareto_rows.append(row_i)

    pareto_df = pd.DataFrame(pareto_rows)

    return pareto_df.sort_values(
        ["n_sensors", "val_macro_f1"],
        ascending=[True, False]
    ).reset_index(drop=True)

In [ ]:
# ============================================================
# MOPSO robustness
# Run one MOPSO seed
# ============================================================

def run_mopso_seed(pso_seed):

    print("\n======================================")
    print("MOPSO seed:", pso_seed)
    print("======================================")

    rng = np.random.default_rng(pso_seed)

    # --------------------------------------------
    # Initial population
    # --------------------------------------------

    positions = np.zeros(
        (N_PARTICLES, n_sensors),
        dtype=int
    )

    initial_sizes = [
        2, 3, 4, 5, 6, 8, 10, 15
    ]

    for i, k in enumerate(initial_sizes):

        selected = rng.choice(
            n_sensors,
            size=k,
            replace=False
        )

        positions[i, selected] = 1


    velocities = rng.uniform(
        -1,
        1,
        size=positions.shape
    )


    pbest_positions = positions.copy()

    pbest_f1 = np.zeros(N_PARTICLES)
    pbest_k = np.zeros(
        N_PARTICLES,
        dtype=int
    )


    run_results = []
    history = []


    # --------------------------------------------
    # Evaluate initial population
    # --------------------------------------------

    for i in range(N_PARTICLES):

        result = evaluate_sensor_subset(
            positions[i],
            use_cache=True
        )

        pbest_f1[i] = result["val_macro_f1"]
        pbest_k[i] = result["n_sensors"]

        run_results.append(
            result.copy()
        )


    # --------------------------------------------
    # MOPSO iterations
    # --------------------------------------------

    for iteration in range(N_ITERATIONS):

        print(
            f"\nMOPSO seed {pso_seed} | "
            f"Iteration {iteration + 1}/{N_ITERATIONS}"
        )


        run_df = pd.DataFrame(run_results)

        pareto_df = get_run_pareto(
            run_df
        )


        if N_ITERATIONS > 1:

            w = (
                W_MAX
                - (W_MAX - W_MIN)
                * iteration
                / (N_ITERATIONS - 1)
            )

        else:
            w = W_MIN


        for i in range(N_PARTICLES):

            # ------------------------------------
            # Choose leader from this run's Pareto
            # ------------------------------------

            available_k = pareto_df[
                "n_sensors"
            ].unique()

            leader_k = rng.choice(
                available_k
            )

            possible_leaders = pareto_df[
                pareto_df["n_sensors"]
                == leader_k
            ]

            leader_index = rng.integers(
                len(possible_leaders)
            )

            leader_row = (
                possible_leaders
                .iloc[leader_index]
            )


            leader_position = np.array(
                [
                    int(x)
                    for x in str(
                        leader_row["mask_key"]
                    ).zfill(n_sensors)
                ]
            )


            # ------------------------------------
            # Binary PSO update
            # ------------------------------------

            r1 = rng.random(n_sensors)
            r2 = rng.random(n_sensors)

            velocities[i] = (
                w * velocities[i]
                + C1 * r1
                * (
                    pbest_positions[i]
                    - positions[i]
                )
                + C2 * r2
                * (
                    leader_position
                    - positions[i]
                )
            )


            velocities[i] = np.clip(
                velocities[i],
                -4,
                4
            )


            probability = (
                1 /
                (1 + np.exp(-velocities[i]))
            )


            new_position = (
                rng.random(n_sensors)
                < probability
            ).astype(int)


            if new_position.sum() == 0:

                random_sensor = rng.integers(
                    n_sensors
                )

                new_position[
                    random_sensor
                ] = 1


            positions[i] = new_position


            # ------------------------------------
            # Evaluate subset
            # ------------------------------------

            result = evaluate_sensor_subset(
                new_position,
                use_cache=True
            )

            run_results.append(
                result.copy()
            )


            new_f1 = result[
                "val_macro_f1"
            ]

            new_k = result[
                "n_sensors"
            ]


            # ------------------------------------
            # Update personal best
            # ------------------------------------

            new_dominates_old = dominates(
                new_f1,
                new_k,
                pbest_f1[i],
                pbest_k[i]
            )

            old_dominates_new = dominates(
                pbest_f1[i],
                pbest_k[i],
                new_f1,
                new_k
            )


            if new_dominates_old:

                pbest_positions[i] = (
                    new_position.copy()
                )

                pbest_f1[i] = new_f1
                pbest_k[i] = new_k


            elif not old_dominates_new:

                if rng.random() < 0.5:

                    pbest_positions[i] = (
                        new_position.copy()
                    )

                    pbest_f1[i] = new_f1
                    pbest_k[i] = new_k


        # ----------------------------------------
        # Pareto after iteration
        # ----------------------------------------

        run_df = pd.DataFrame(
            run_results
        )

        pareto_df = get_run_pareto(
            run_df
        )


        history.append({
            "iteration": iteration + 1,
            "unique_subsets":
                run_df["mask_key"].nunique(),
            "pareto_size":
                len(pareto_df)
        })


        print("\nCurrent Pareto:")

        print(
            pareto_df[
                [
                    "n_sensors",
                    "val_macro_f1",
                    "sensors"
                ]
            ].to_string(index=False)
        )


        # Save after each iteration
        run_df.to_csv(
            PSO_RESULTS_DIR /
            f"chronological_MOPSO_15sensor_"
            f"evaluations_psoseed{pso_seed}.csv",
            index=False
        )

        pareto_df.to_csv(
            PSO_RESULTS_DIR /
            f"chronological_MOPSO_15sensor_"
            f"pareto_psoseed{pso_seed}.csv",
            index=False
        )


    history_df = pd.DataFrame(
        history
    )


    history_df.to_csv(
        PSO_RESULTS_DIR /
        f"chronological_MOPSO_15sensor_"
        f"history_psoseed{pso_seed}.csv",
        index=False
    )


    return pareto_df

In [ ]:
# ============================================================
# MOPSO robustness
# Run three PSO seeds
# ============================================================
MOPSO_SEEDS = [7,123,2026]
all_pareto_results = []

for pso_seed in MOPSO_SEEDS:

    pareto_seed = run_mopso_seed(
        pso_seed
    )

    pareto_seed = pareto_seed.copy()

    pareto_seed[
        "pso_seed"
    ] = pso_seed

    all_pareto_results.append(
        pareto_seed
    )


all_pareto_df = pd.concat(
    all_pareto_results,
    ignore_index=True
)


MOPSO seed: 7
Cached: ['DA10', 'DA14'] | Seed: 42 | Val F1: 0.3922219024968669
Cached: ['DA09', 'DA12', 'DA15'] | Seed: 42 | Val F1: 0.8184369073142362
Cached: ['DA01', 'DA04', 'DA14', 'DA15'] | Seed: 42 | Val F1: 0.9242514549796628
Cached: ['DA02', 'DA08', 'DA10', 'DA11', 'DA14'] | Seed: 42 | Val F1: 0.8321274510184942
Cached: ['DA03', 'DA06', 'DA07', 'DA08', 'DA12', 'DA15'] | Seed: 42 | Val F1: 0.9893130813779312
Cached: ['DA04', 'DA07', 'DA09', 'DA10', 'DA12', 'DA13', 'DA14', 'DA15'] | Seed: 42 | Val F1: 0.9890316554867492
Cached: ['DA01', 'DA04', 'DA05', 'DA07', 'DA08', 'DA09', 'DA10', 'DA11', 'DA13', 'DA15'] | Seed: 42 | Val F1: 0.994844955963514
Cached: ['DA01', 'DA02', 'DA03', 'DA04', 'DA05', 'DA06', 'DA07', 'DA08', 'DA09', 'DA10', 'DA11', 'DA12', 'DA13', 'DA14', 'DA15'] | Seed: 42 | Val F1: 1.0

MOPSO seed 7 | Iteration 1/5
Cached: ['DA01', 'DA05', 'DA07', 'DA08', 'DA11', 'DA12', 'DA13', 'DA14', 'DA15'] | Seed: 42 | Val F1: 0.9983659247244
Cached: ['DA01', 'DA05', 'DA07', 'DA0

In [ ]:
# ============================================================
# MOPSO robustness
# Compare PSO seeds
# ============================================================

comparison = (all_pareto_df.groupby(["pso_seed","n_sensors"])["val_macro_f1"] .max() .reset_index())

print(comparison.to_string(index=False ))

 pso_seed  n_sensors  val_macro_f1
        7          2      0.763371
        7          3      0.897325
        7          4      0.966377
        7          5      0.993923
        7          6      0.996479
        7          7      0.998366
        7          8      1.000000
      123          1      0.332370
      123          2      0.701363
      123          3      0.948689
      123          4      0.993234
      123          5      0.998366
      123          7      1.000000
     2026          2      0.678716
     2026          3      0.932993
     2026          4      0.972753
     2026          5      0.986854
     2026          6      1.000000


In [ ]:
# ============================================================
# Candidate validation
# Fixed candidates, seeds and output paths
# ============================================================

import json

assert RUN_TEST is False

assert list(selected_sensor_names) == [
    f"DA{i:02d}" for i in range(1, 16)
]

assert (
    X_train_tensor.shape[2]
    == X_val_tensor.shape[2]
    == 15
)

assert list(MODEL_SEEDS) == [42, 7, 123, 2026]


final_candidates = {

    "Floor_4": [
        "DA05", "DA09", "DA13", "DA15"
    ],

    "WithBase_4": [
        "DA01", "DA09", "DA12", "DA13"
    ],

    "Floor_5": [
        "DA05", "DA06", "DA07", "DA08", "DA13"
    ],

    "WithBase_5": [
        "DA01", "DA05", "DA06", "DA13", "DA15"
    ],

    "WithBase_6": [
        "DA01", "DA03", "DA05",
        "DA09", "DA13", "DA15"
    ],

    "Full_15": list(selected_sensor_names),

    "Full_12": [
        f"DA{i:02d}" for i in range(4, 16)
    ]
}


MAX_MEAN_F1_DROP = 0.01

FINAL_DIR = PSO_RESULTS_DIR / "final_candidates_v1"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_CACHE_PATH = FINAL_DIR / "candidate_cache.csv"
FINAL_MODELS_DIR = FINAL_DIR / "best_models"


# Record the existing settings
settings = {

    "candidates": final_candidates,
    "model_seeds": list(MODEL_SEEDS),

    "reference": "Full_15",
    "max_mean_f1_drop": MAX_MEAN_F1_DROP,

    "train_shape": list(X_train_tensor.shape),
    "val_shape": list(X_val_tensor.shape),

    "preprocessing": [
        FS_DASY, TRAIN_RATIO, VAL_RATIO,
        CUTOFF, ORDER, EDGE_SEC, GAP_SEC,
        WINDOW_SIZE, STEP_SIZE
    ],

    "model": [
        D_MODEL, NUM_HEADS, NUM_LAYERS,
        DIM_FEEDFORWARD, DROPOUT
    ],

    "training": [
        LEARNING_RATE, WEIGHT_DECAY, BATCH_SIZE,
        NUM_EPOCHS, PATIENCE, MIN_DELTA, GRAD_CLIP
    ]
}


settings_path = FINAL_DIR / "settings.json"

if settings_path.exists():

    if json.loads(settings_path.read_text()) != settings:
        raise ValueError(
            "Settings changed. Do not mix these runs in one cache."
        )

else:
    settings_path.write_text(
        json.dumps(settings, indent=2)
    )


print("Candidates:", len(final_candidates))
print("Seeds:", MODEL_SEEDS)

print(
    "Total required results:",
    len(final_candidates) * len(MODEL_SEEDS)
)

print("New results folder:", FINAL_DIR)

Candidates: 7
Seeds: [42, 7, 123, 2026]
Total required results: 28
New results folder: /content/drive/MyDrive/ASCE_IASC/PSO_results/final_candidates_v1


In [ ]:
# ============================================================
# Candidate validation
# Import compatible cached results
# ============================================================

source_paths = [

    # Optional old 12-sensor cache
    PSO_RESULTS_DIR
    / "chronological_PSO_subset_cache_seed42.csv",

    PSO_RESULTS_DIR
    / "chronological_PSO_15sensor_cache_seed42.csv",

    PSO_RESULTS_DIR
    / "chronological_phase3_model_seed_cache.csv",

    # Keep results already completed in this stage
    FINAL_CACHE_PATH
]


# Required candidate-seed combinations
plan_rows = []

for name, sensors in final_candidates.items():

    mask = sensor_names_to_mask(sensors)
    _, ordered_sensors, key = get_sensor_subset(mask)

    for seed in MODEL_SEEDS:

        plan_rows.append({
            "candidate": name,
            "sensors": "|".join(ordered_sensors),
            "n_sensors": len(sensors),
            "mask_key": key,
            "model_seed": seed
        })

plan_df = pd.DataFrame(plan_rows)


# Read existing results
cache_parts = []

for path in source_paths:

    if not path.exists():
        print("Not found, skipped:", path.name)
        continue

    df = pd.read_csv(
        path,
        dtype={"mask_key": str}
    )

    df["sensors"] = df["sensors"].apply(
        lambda text: "|".join(
            sorted(s.strip() for s in text.split("|"))
        )
    )

    df = df[
        df["sensors"].isin(plan_df["sensors"])
        & df["model_seed"].isin(MODEL_SEEDS)
    ].copy()

    if df.empty:
        continue

    # Build every mask in the current 15-channel order
    df["mask_key"] = df["sensors"].apply(
        lambda text: get_sensor_subset(
            sensor_names_to_mask(text.split("|"))
        )[2]
    )

    if "source_file" not in df.columns:
        df["source_file"] = path.name

    cache_parts.append(df)


if not cache_parts:
    raise FileNotFoundError(
        "No candidate results found. Check the cache paths."
    )


candidate_cache = pd.concat(
    cache_parts,
    ignore_index=True
)

if not candidate_cache["val_macro_f1"].between(0, 1).all():
    raise ValueError(
        "Invalid or missing F1 in the input cache."
    )


# Check conflicting records
check = candidate_cache.groupby(
    ["mask_key", "model_seed"]
).agg(
    f1_min=("val_macro_f1", "min"),
    f1_max=("val_macro_f1", "max"),
    epoch_count=("best_epoch", "nunique")
)

if (
    (check["f1_max"] - check["f1_min"] > 1e-10)
    | (check["epoch_count"] > 1)
).any():

    raise ValueError(
        "Conflicting cached results. Stop and inspect them."
    )


candidate_cache = candidate_cache.drop_duplicates(
    ["mask_key", "model_seed"],
    keep="last"
)

if "weights_path" not in candidate_cache.columns:
    candidate_cache["weights_path"] = ""

candidate_cache["weights_path"] = (
    candidate_cache["weights_path"].fillna("")
)


# Write only to the new candidate cache
temp_path = FINAL_CACHE_PATH.with_suffix(".tmp")

candidate_cache.to_csv(temp_path, index=False)
temp_path.replace(FINAL_CACHE_PATH)


# Show remaining work
status_df = plan_df.merge(
    candidate_cache[
        ["mask_key", "model_seed", "val_macro_f1"]
    ],
    on=["mask_key", "model_seed"],
    how="left",
    validate="one_to_one"
)

missing_df = status_df[
    status_df["val_macro_f1"].isna()
]

print(
    "\nAvailable results:",
    len(status_df) - len(missing_df)
)

print("New trainings needed:", len(missing_df))

print(
    missing_df[
        ["candidate", "model_seed"]
    ].to_string(index=False)
)


Available results: 28
New trainings needed: 0
Empty DataFrame
Columns: [candidate, model_seed]
Index: []


In [ ]:
# ============================================================
# Candidate validation
# Run missing candidate-seed evaluations
# ============================================================

assert RUN_TEST is False

# Avoid accidentally starting missing trainings on CPU
if len(missing_df) and (
    not torch.cuda.is_available()
    or torch.device(device).type != "cuda"
):
    raise RuntimeError(
        "GPU is not ready. "
        "Cells 1 and 2 are safe; run Cell 3 later."
    )


previous_seed = MODEL_SEED
previous_cache_path = CACHE_PATH

try:

    CACHE_PATH = FINAL_CACHE_PATH

    for name, sensors in final_candidates.items():

        print("\n==========", name, "==========")

        mask = sensor_names_to_mask(sensors)

        for seed in MODEL_SEEDS:

            MODEL_SEED = seed

            result = evaluate_sensor_subset(
                mask,
                use_cache=True,
                checkpoint_dir=FINAL_MODELS_DIR
            )

finally:

    MODEL_SEED = previous_seed
    CACHE_PATH = previous_cache_path


print("\nCandidate evaluation completed.")


========== Floor_4 ==========
Cached: ['DA05', 'DA09', 'DA13', 'DA15'] | Seed: 42 | Val F1: 0.9930719182959636
Cached: ['DA05', 'DA09', 'DA13', 'DA15'] | Seed: 7 | Val F1: 0.9832674666423636
Cached: ['DA05', 'DA09', 'DA13', 'DA15'] | Seed: 123 | Val F1: 0.9867233133968992
Cached: ['DA05', 'DA09', 'DA13', 'DA15'] | Seed: 2026 | Val F1: 0.9865250011449972

========== WithBase_4 ==========
Cached: ['DA01', 'DA09', 'DA12', 'DA13'] | Seed: 42 | Val F1: 0.9932335165722536
Cached: ['DA01', 'DA09', 'DA12', 'DA13'] | Seed: 7 | Val F1: 0.9947059935715636
Cached: ['DA01', 'DA09', 'DA12', 'DA13'] | Seed: 123 | Val F1: 1.0
Cached: ['DA01', 'DA09', 'DA12', 'DA13'] | Seed: 2026 | Val F1: 0.9983659247244

========== Floor_5 ==========
Cached: ['DA05', 'DA06', 'DA07', 'DA08', 'DA13'] | Seed: 42 | Val F1: 0.9983659247244
Cached: ['DA05', 'DA06', 'DA07', 'DA08', 'DA13'] | Seed: 7 | Val F1: 0.9983659247244
Cached: ['DA05', 'DA06', 'DA07', 'DA08', 'DA13'] | Seed: 123 | Val F1: 0.9877598641183396
Cached: [

In [ ]:
# ============================================================
# Prepare missing Full-15 weights for Importance
# ============================================================

assert RUN_TEST is False

# Keep rebuilt results separate from the original scores
rebuild_dir = FINAL_DIR / "full15_rebuild"
rebuild_dir.mkdir(parents=True, exist_ok=True)

FINAL_MODELS_DIR.mkdir(parents=True, exist_ok=True)

scores = pd.read_csv(
    FINAL_CACHE_PATH,
    dtype={"mask_key": str, "weights_path": str}
)

scores["weights_path"] = scores["weights_path"].fillna("")

full_mask = sensor_names_to_mask(
    final_candidates["Full_15"]
)

_, _, full_key = get_sensor_subset(full_mask)

old_seed = MODEL_SEED
old_cache_path = CACHE_PATH

try:

    CACHE_PATH = rebuild_dir / "rebuild_cache.csv"

    for seed in MODEL_SEEDS:

        final_path = (
            FINAL_MODELS_DIR / f"{full_key}_seed{seed}.pt"
        )

        if final_path.is_file():
            print("Full-15 weights already exist. Seed:", seed)
            continue

        selected_row = (
            (scores["mask_key"] == full_key)
            & (scores["model_seed"] == seed)
        )

        if selected_row.sum() != 1:
            raise ValueError(
                "Complete candidate validation first."
            )

        old_result = scores.loc[selected_row].iloc[0]
        staged_path = rebuild_dir / final_path.name

        if (
            not staged_path.is_file()
            and torch.device(device).type != "cuda"
        ):
            raise RuntimeError(
                "GPU is not ready for this training."
            )

        MODEL_SEED = seed

        result = evaluate_sensor_subset(
            full_mask,
            use_cache=staged_path.is_file(),
            checkpoint_dir=rebuild_dir
        )

        same_f1 = np.isclose(
            result["val_macro_f1"],
            old_result["val_macro_f1"],
            atol=1e-6,
            rtol=0
        )

        same_epoch = (
            result["best_epoch"] == old_result["best_epoch"]
        )

        if not (same_f1 and same_epoch):
            raise RuntimeError(
                f"Seed {seed}: rebuilt result differs. "
                "Original candidate scores were not changed."
            )

        # Move the checked weights to the Importance model folder
        Path(result["weights_path"]).replace(final_path)

        # Update only the recorded weights path
        scores.loc[selected_row, "weights_path"] = str(final_path)

        temp_path = FINAL_CACHE_PATH.with_suffix(".tmp")
        scores.to_csv(temp_path, index=False)
        temp_path.replace(FINAL_CACHE_PATH)

        print("Full-15 weights prepared. Seed:", seed)

finally:

    MODEL_SEED = old_seed
    CACHE_PATH = old_cache_path

Full-15 weights already exist. Seed: 42
Full-15 weights already exist. Seed: 7
Full-15 weights already exist. Seed: 123
Full-15 weights already exist. Seed: 2026


In [ ]:
# ============================================================
# Candidate validation
# Paired comparison with Full-15
# ============================================================

candidate_cache = pd.read_csv(
    FINAL_CACHE_PATH,
    dtype={"mask_key": str}
)

final_results = plan_df.merge(
    candidate_cache[
        [
            "mask_key",
            "model_seed",
            "val_macro_f1",
            "best_epoch",
            "epochs_run",
            "training_seconds",
            "weights_path"
        ]
    ],
    on=["mask_key", "model_seed"],
    how="left",
    validate="one_to_one"
)


if final_results["val_macro_f1"].isna().any():

    raise RuntimeError(
        "Some results are missing. "
        "Do not compare incomplete seed sets."
    )


# Full-15 score for each model seed
full15_scores = final_results[
    final_results["candidate"] == "Full_15"
].set_index("model_seed")["val_macro_f1"]


final_results["full15_f1"] = (
    final_results["model_seed"].map(full15_scores)
)

final_results["f1_drop"] = (
    final_results["full15_f1"]
    - final_results["val_macro_f1"]
)


# Summary across the four seeds
summary = final_results.groupby(
    ["candidate", "n_sensors"]
).agg(
    n_seeds=("model_seed", "nunique"),

    mean_f1=("val_macro_f1", "mean"),
    std_f1=("val_macro_f1", "std"),
    min_f1=("val_macro_f1", "min"),
    max_f1=("val_macro_f1", "max"),

    mean_drop=("f1_drop", "mean"),
    worst_drop=("f1_drop", "max")
).reset_index()


summary["within_mean_drop"] = (
    summary["mean_drop"]
    <= MAX_MEAN_F1_DROP + 1e-12
)

summary = summary.sort_values(
    ["n_sensors", "mean_f1"],
    ascending=[True, False]
)


# Table 1: results for each seed
print("\nValidation F1 by model seed:")

print(
    final_results.pivot(
        index="candidate",
        columns="model_seed",
        values="val_macro_f1"
    )
    .reindex(columns=MODEL_SEEDS)
    .to_string(float_format=lambda x: f"{x:.6f}")
)


# Table 2: summary and paired differences
print("\nSummary and paired differences from Full-15:")

print(
    summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


final_results.to_csv(
    FINAL_DIR / "results_by_seed.csv",
    index=False
)

summary.to_csv(
    FINAL_DIR / "summary.csv",
    index=False
)

print("\nSaved to:", FINAL_DIR)


Validation F1 by model seed:
model_seed     42       7        123      2026
candidate                                     
Floor_4    0.993072 0.983267 0.986723 0.986525
Floor_5    0.998366 0.998366 0.987760 0.989394
Full_12    1.000000 1.000000 1.000000 1.000000
Full_15    1.000000 0.998366 1.000000 1.000000
WithBase_4 0.993234 0.994706 1.000000 0.998366
WithBase_5 0.998366 0.989792 0.987760 0.950864
WithBase_6 1.000000 0.995098 0.995098 0.991200

Summary and paired differences from Full-15:
 candidate  n_sensors  n_seeds  mean_f1   std_f1   min_f1   max_f1  mean_drop  worst_drop  within_mean_drop
WithBase_4          4        4 0.996576 0.003141 0.993234 1.000000   0.003015    0.006766              True
   Floor_4          4        4 0.987397 0.004102 0.983267 0.993072   0.012195    0.015098             False
   Floor_5          5        4 0.993471 0.005691 0.987760 0.998366   0.006120    0.012240              True
WithBase_5          5        4 0.981695 0.021062 0.950864 0.998366   

# Importance Top_k validation

In [ ]:
# ============================================================
# Importance Top-k validation
# Define sensor subsets
# ============================================================

importance_candidates = {

    "Importance_4": ["DA15", "DA06", "DA13", "DA12" ],

    "Importance_5": ["DA15", "DA06", "DA13", "DA12", "DA14"],

    "Importance_6": ["DA15", "DA06", "DA13", "DA12", "DA14", "DA11"]
}


for name, sensors in importance_candidates.items():

    print( name,"->", len(sensors),"sensors:", sensors )

Importance_4 -> 4 sensors: ['DA15', 'DA06', 'DA13', 'DA12']
Importance_5 -> 5 sensors: ['DA15', 'DA06', 'DA13', 'DA12', 'DA14']
Importance_6 -> 6 sensors: ['DA15', 'DA06', 'DA13', 'DA12', 'DA14', 'DA11']


In [ ]:
# ============================================================
# Importance Top-k validation
# Save folder and cache
# ============================================================

TOPK_DIR = (PSO_RESULTS_DIR / "importance_topk_validation")

TOPK_DIR.mkdir(parents=True, exist_ok=True)

TOPK_CACHE_PATH = (TOPK_DIR / "importance_topk_cache.csv")

print("Top-k results folder:")
print(TOPK_DIR)

Top-k results folder:
/content/drive/MyDrive/ASCE_IASC/PSO_results/importance_topk_validation


In [ ]:
# ============================================================
# Importance Top-k validation
# Train each subset with four model seeds
# ============================================================

assert RUN_TEST is False

previous_seed = MODEL_SEED
previous_cache_path = CACHE_PATH

topk_results = []

try:

    CACHE_PATH = TOPK_CACHE_PATH

    for name, sensors in importance_candidates.items():

        print("\n============================")
        print(name)
        print("Sensors:", sensors)
        print("============================")

        mask = sensor_names_to_mask(sensors)

        for seed in MODEL_SEEDS:

            print("\nModel seed:", seed)

            MODEL_SEED = seed

            result = evaluate_sensor_subset(mask,use_cache=True )

            topk_results.append({

                "candidate": name,
                "sensors": result["sensors"],
                "n_sensors": result["n_sensors"],
                "model_seed": seed,
                "val_macro_f1": result["val_macro_f1"],
                "best_epoch": result["best_epoch"]
            })

finally:

    MODEL_SEED = previous_seed
    CACHE_PATH = previous_cache_path


print("\nTop-k evaluation completed.")


Importance_4
Sensors: ['DA15', 'DA06', 'DA13', 'DA12']

Model seed: 42
Cached: ['DA06', 'DA12', 'DA13', 'DA15'] | Seed: 42 | Val F1: 0.9210387488328664

Model seed: 7
Cached: ['DA06', 'DA12', 'DA13', 'DA15'] | Seed: 7 | Val F1: 0.928275719989538

Model seed: 123
Cached: ['DA06', 'DA12', 'DA13', 'DA15'] | Seed: 123 | Val F1: 0.9324887922941444

Model seed: 2026
Cached: ['DA06', 'DA12', 'DA13', 'DA15'] | Seed: 2026 | Val F1: 0.9588404849669524

Importance_5
Sensors: ['DA15', 'DA06', 'DA13', 'DA12', 'DA14']

Model seed: 42
Cached: ['DA06', 'DA12', 'DA13', 'DA14', 'DA15'] | Seed: 42 | Val F1: 0.9343921903528816

Model seed: 7
Cached: ['DA06', 'DA12', 'DA13', 'DA14', 'DA15'] | Seed: 7 | Val F1: 0.965334779223668

Model seed: 123
Cached: ['DA06', 'DA12', 'DA13', 'DA14', 'DA15'] | Seed: 123 | Val F1: 0.9362146884097982

Model seed: 2026
Cached: ['DA06', 'DA12', 'DA13', 'DA14', 'DA15'] | Seed: 2026 | Val F1: 0.920414071023827

Importance_6
Sensors: ['DA15', 'DA06', 'DA13', 'DA12', 'DA14', 'DA

In [ ]:
# ============================================================
# Importance Top-k validation
# Summary
# ============================================================

topk_results_df = pd.DataFrame(topk_results)


topk_summary = (
    topk_results_df
    .groupby(
        ["candidate", "n_sensors"]
    )
    .agg(
        mean_f1=("val_macro_f1", "mean"),
        std_f1=("val_macro_f1", "std"),
        min_f1=("val_macro_f1", "min"),
        max_f1=("val_macro_f1", "max")
    )
    .reset_index()
)


print("\nValidation F1 by model seed:")

display(
    topk_results_df.pivot(
        index="candidate",
        columns="model_seed",
        values="val_macro_f1"
    )
)


print("\nSummary:")

display(topk_summary)


Validation F1 by model seed:


model_seed,7,42,123,2026
candidate,,,,
Importance_4,0.928276,0.921039,0.932489,0.958840
Importance_5,0.965335,0.934392,0.936215,0.920414
Importance_6,0.992834,0.978687,0.977285,0.984022



Summary:


,candidate,n_sensors,mean_f1,std_f1,min_f1,max_f1
0,Importance_4,4,0.935161,0.016479,0.921039,0.958840
1,Importance_5,5,0.939089,0.018867,0.920414,0.965335
2,Importance_6,6,0.983207,0.007044,0.977285,0.992834


In [ ]:
topk_results_df.to_csv(TOPK_DIR / "results_by_seed.csv",index=False)

topk_summary.to_csv(TOPK_DIR / "summary.csv",index=False)

print("Results saved to:")
print(TOPK_DIR)

Results saved to:
/content/drive/MyDrive/ASCE_IASC/PSO_results/importance_topk_validation


In [ ]:
# ============================================================
# Bottom-k importance sensor subsets
# ============================================================

bottom_candidates = {

    "Bottom_4": ["DA10", "DA02", "DA04", "DA01"],

    "Bottom_5": ["DA10", "DA02", "DA04", "DA01", "DA08"]}


for name, sensors in bottom_candidates.items():

    print(name,"->", len(sensors),"sensors:", sensors)

Bottom_4 -> 4 sensors: ['DA10', 'DA02', 'DA04', 'DA01']
Bottom_5 -> 5 sensors: ['DA10', 'DA02', 'DA04', 'DA01', 'DA08']


In [ ]:
# ============================================================
# Bottom-k validation folder
# ============================================================

BOTTOM_DIR = (PSO_RESULTS_DIR / "importance_bottomk_validation")

BOTTOM_DIR.mkdir(parents=True,exist_ok=True)

BOTTOM_CACHE_PATH = (BOTTOM_DIR / "importance_bottomk_cache.csv")

print("Bottom-k results folder:")
print(BOTTOM_DIR)

Bottom-k results folder:
/content/drive/MyDrive/ASCE_IASC/PSO_results/importance_bottomk_validation


In [ ]:
# ============================================================
# Train Bottom-4 and Bottom-5
# ============================================================

assert RUN_TEST is False

previous_seed = MODEL_SEED
previous_cache_path = CACHE_PATH

bottom_results = []

try:

    CACHE_PATH = BOTTOM_CACHE_PATH

    for name, sensors in bottom_candidates.items():

        print("\n============================")
        print(name)
        print("Sensors:", sensors)
        print("============================")

        mask = sensor_names_to_mask(sensors)

        for seed in MODEL_SEEDS:

            print("\nModel seed:", seed)

            MODEL_SEED = seed

            result = evaluate_sensor_subset( mask, use_cache=True )

            bottom_results.append({

                "candidate": name,
                "sensors": result["sensors"],
                "n_sensors": result["n_sensors"],
                "model_seed": seed,
                "val_macro_f1": result["val_macro_f1"],
                "best_epoch": result["best_epoch"]
            })

finally:

    MODEL_SEED = previous_seed
    CACHE_PATH = previous_cache_path


print("\nBottom-k evaluation completed.")


Bottom_4
Sensors: ['DA10', 'DA02', 'DA04', 'DA01']

Model seed: 42
Cached: ['DA01', 'DA02', 'DA04', 'DA10'] | Seed: 42 | Val F1: 0.7581417816462745

Model seed: 7
Cached: ['DA01', 'DA02', 'DA04', 'DA10'] | Seed: 7 | Val F1: 0.8208942526063797

Model seed: 123
Cached: ['DA01', 'DA02', 'DA04', 'DA10'] | Seed: 123 | Val F1: 0.8388898152740288

Model seed: 2026
Cached: ['DA01', 'DA02', 'DA04', 'DA10'] | Seed: 2026 | Val F1: 0.8750873935683052

Bottom_5
Sensors: ['DA10', 'DA02', 'DA04', 'DA01', 'DA08']

Model seed: 42
Cached: ['DA01', 'DA02', 'DA04', 'DA08', 'DA10'] | Seed: 42 | Val F1: 0.9427466975851132

Model seed: 7
Cached: ['DA01', 'DA02', 'DA04', 'DA08', 'DA10'] | Seed: 7 | Val F1: 0.9685055009427572

Model seed: 123
Cached: ['DA01', 'DA02', 'DA04', 'DA08', 'DA10'] | Seed: 123 | Val F1: 0.9671109901731068

Model seed: 2026
Cached: ['DA01', 'DA02', 'DA04', 'DA08', 'DA10'] | Seed: 2026 | Val F1: 0.9574978352324032

Bottom-k evaluation completed.


In [ ]:
# ============================================================
# Bottom-k summary
# ============================================================

bottom_results_df = pd.DataFrame(bottom_results)


bottom_summary = (
    bottom_results_df.groupby(["candidate", "n_sensors"]
    ).agg(
        mean_f1=("val_macro_f1", "mean"),
        std_f1=("val_macro_f1", "std"),
        min_f1=("val_macro_f1", "min"),
        max_f1=("val_macro_f1", "max")
    ).reset_index())


print("\nValidation F1 by model seed:")
display(bottom_results_df.pivot(index="candidate",columns="model_seed",values="val_macro_f1"))


print("\nSummary:")
display(bottom_summary)


Validation F1 by model seed:


model_seed,7,42,123,2026
candidate,,,,
Bottom_4,0.820894,0.758142,0.838890,0.875087
Bottom_5,0.968506,0.942747,0.967111,0.957498



Summary:


,candidate,n_sensors,mean_f1,std_f1,min_f1,max_f1
0,Bottom_4,4,0.823253,0.048909,0.758142,0.875087
1,Bottom_5,5,0.958965,0.011868,0.942747,0.968506


In [ ]:
# ============================================================
# Random Search settings
# ============================================================

RANDOM_SEARCH_SEED = 2026

K_VALUES = [4, 5, 6]

N_RANDOM_PER_K = 10

RANDOM_DIR = (PSO_RESULTS_DIR / "random_search")
RANDOM_DIR.mkdir(parents=True,exist_ok=True)
RANDOM_CACHE_PATH = (RANDOM_DIR / "random_search_cache.csv")

print("Random search folder:")
print(RANDOM_DIR)

Random search folder:
/content/drive/MyDrive/ASCE_IASC/PSO_results/random_search


In [ ]:
# ============================================================
# Create random sensor subsets
# ============================================================

rng = np.random.default_rng(RANDOM_SEARCH_SEED)

random_subsets = []


for k in K_VALUES:
    seen = set()

    while len(seen) < N_RANDOM_PER_K:

        sensors = rng.choice(selected_sensor_names,size=k,replace=False).tolist()
        sensors = sorted(sensors)
        key = "|".join(sensors)

        if key not in seen:

            seen.add(key)
            random_subsets.append({"n_sensors": k, "sensors": sensors })


print("Number of random subsets:", len(random_subsets))


for item in random_subsets:

    print(
        item["n_sensors"],
        item["sensors"])

Number of random subsets: 30
4 ['DA01', 'DA03', 'DA10', 'DA11']
4 ['DA05', 'DA09', 'DA13', 'DA14']
4 ['DA03', 'DA09', 'DA10', 'DA13']
4 ['DA05', 'DA10', 'DA12', 'DA13']
4 ['DA02', 'DA07', 'DA10', 'DA13']
4 ['DA03', 'DA04', 'DA05', 'DA15']
4 ['DA02', 'DA04', 'DA06', 'DA10']
4 ['DA03', 'DA06', 'DA13', 'DA14']
4 ['DA04', 'DA05', 'DA06', 'DA10']
4 ['DA08', 'DA11', 'DA12', 'DA13']
5 ['DA06', 'DA08', 'DA09', 'DA11', 'DA12']
5 ['DA04', 'DA05', 'DA06', 'DA10', 'DA15']
5 ['DA03', 'DA05', 'DA07', 'DA08', 'DA15']
5 ['DA07', 'DA08', 'DA09', 'DA10', 'DA12']
5 ['DA01', 'DA06', 'DA07', 'DA09', 'DA12']
5 ['DA01', 'DA05', 'DA10', 'DA12', 'DA14']
5 ['DA02', 'DA03', 'DA04', 'DA06', 'DA08']
5 ['DA01', 'DA04', 'DA05', 'DA10', 'DA11']
5 ['DA03', 'DA05', 'DA07', 'DA08', 'DA11']
5 ['DA02', 'DA03', 'DA04', 'DA07', 'DA15']
6 ['DA01', 'DA02', 'DA04', 'DA08', 'DA09', 'DA10']
6 ['DA06', 'DA07', 'DA08', 'DA09', 'DA11', 'DA15']
6 ['DA02', 'DA03', 'DA04', 'DA12', 'DA13', 'DA14']
6 ['DA04', 'DA06', 'DA09', 'DA12', 'DA

In [ ]:
# ============================================================
# Random Search
# Model seed is fixed to 42
# ============================================================

assert RUN_TEST is False

previous_seed = MODEL_SEED
previous_cache_path = CACHE_PATH

random_search_results = []

try:

    MODEL_SEED = 42

    CACHE_PATH = RANDOM_CACHE_PATH


    for i, item in enumerate(random_subsets,start=1):

        k = item["n_sensors"]
        sensors = item["sensors"]

        print("\n============================")
        print(f"Random subset {i}/"f"{len(random_subsets)}")
        print("k =", k)
        print("Sensors:", sensors)
        print("============================")

        mask = sensor_names_to_mask(sensors)

        result = evaluate_sensor_subset(mask,use_cache=True)

        random_search_results.append({

            "n_sensors": k,
            "sensors": result["sensors"],
            "model_seed": 42,
            "val_macro_f1": result["val_macro_f1"],
            "best_epoch": result["best_epoch"]})


finally:

    MODEL_SEED = previous_seed
    CACHE_PATH = previous_cache_path


print("\nRandom Search completed.")


Random subset 1/30
k = 4
Sensors: ['DA01', 'DA03', 'DA10', 'DA11']
Cached: ['DA01', 'DA03', 'DA10', 'DA11'] | Seed: 42 | Val F1: 0.8984442196965963

Random subset 2/30
k = 4
Sensors: ['DA05', 'DA09', 'DA13', 'DA14']
Cached: ['DA05', 'DA09', 'DA13', 'DA14'] | Seed: 42 | Val F1: 0.9655250489515176

Random subset 3/30
k = 4
Sensors: ['DA03', 'DA09', 'DA10', 'DA13']
Cached: ['DA03', 'DA09', 'DA10', 'DA13'] | Seed: 42 | Val F1: 0.8903182362978315

Random subset 4/30
k = 4
Sensors: ['DA05', 'DA10', 'DA12', 'DA13']
Cached: ['DA05', 'DA10', 'DA12', 'DA13'] | Seed: 42 | Val F1: 0.9646997029404208

Random subset 5/30
k = 4
Sensors: ['DA02', 'DA07', 'DA10', 'DA13']
Cached: ['DA02', 'DA07', 'DA10', 'DA13'] | Seed: 42 | Val F1: 0.9290426096345744

Random subset 6/30
k = 4
Sensors: ['DA03', 'DA04', 'DA05', 'DA15']
Cached: ['DA03', 'DA04', 'DA05', 'DA15'] | Seed: 42 | Val F1: 0.9832426291285716

Random subset 7/30
k = 4
Sensors: ['DA02', 'DA04', 'DA06', 'DA10']
Cached: ['DA02', 'DA04', 'DA06', 'DA10

In [ ]:
# ============================================================
# Best random subset for each k
# ============================================================

random_search_df = pd.DataFrame(random_search_results)


best_random_df = (random_search_df.sort_values(["n_sensors", "val_macro_f1"],
        ascending=[True, False]).groupby("n_sensors", as_index=False ).first())


print("Best random subset for each k:")

display(best_random_df)

Best random subset for each k:


,n_sensors,sensors,model_seed,val_macro_f1,best_epoch
0,4,DA08|DA11|DA12|DA13,42,0.985137,51
1,5,DA01|DA06|DA07|DA09|DA12,42,0.991049,20
2,6,DA04|DA06|DA09|DA12|DA13|DA15,42,0.996479,35


In [ ]:
random_search_df.to_csv(RANDOM_DIR / "random_search_all_subsets.csv",index=False)

best_random_df.to_csv( RANDOM_DIR / "random_search_best_subsets.csv", index=False)

print("Random Search results saved.")

Random Search results saved.


In [ ]:
# ============================================================
# Validate best random subsets
# with four model seeds
# ============================================================

assert RUN_TEST is False

previous_seed = MODEL_SEED
previous_cache_path = CACHE_PATH

random_final_results = []

try:

    CACHE_PATH = RANDOM_CACHE_PATH


    for _, row in best_random_df.iterrows():

        k = int(row["n_sensors"])
        sensors = (row["sensors"].split("|"))
        candidate_name = (f"Random_{k}" )

        print("\n============================")
        print(candidate_name)
        print("Sensors:", sensors)
        print("============================")


        mask = sensor_names_to_mask(sensors)


        for seed in MODEL_SEEDS:

            print("\nModel seed:",seed )

            MODEL_SEED = seed

            result = evaluate_sensor_subset( mask, use_cache=True)

            random_final_results.append({

                "candidate": candidate_name,
                "sensors":  result["sensors"],
                "n_sensors": result["n_sensors"],
                "model_seed": seed,
                "val_macro_f1": result["val_macro_f1"],
                "best_epoch":   result["best_epoch"] })


finally:

    MODEL_SEED = previous_seed
    CACHE_PATH = previous_cache_path


print("\nRandom candidate validation completed.")


Random_4
Sensors: ['DA08', 'DA11', 'DA12', 'DA13']

Model seed: 42
Cached: ['DA08', 'DA11', 'DA12', 'DA13'] | Seed: 42 | Val F1: 0.9851366589985856

Model seed: 7
Cached: ['DA08', 'DA11', 'DA12', 'DA13'] | Seed: 7 | Val F1: 0.9815842543358684

Model seed: 123
Cached: ['DA08', 'DA11', 'DA12', 'DA13'] | Seed: 123 | Val F1: 0.9586173527795836

Model seed: 2026

Training: ['DA08', 'DA11', 'DA12', 'DA13'] | Seed: 2026


/tmp/ipykernel_1577/3525986821.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer,num_layers=num_layers)
/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 001 | Train F1: 0.0831 | Val F1: 0.0985 | Best: 0.0985
Epoch 005 | Train F1: 0.7987 | Val F1: 0.7064 | Best: 0.8211
Epoch 010 | Train F1: 0.9298 | Val F1: 0.8927 | Best: 0.8979
Epoch 015 | Train F1: 0.9579 | Val F1: 0.9388 | Best: 0.9388
Epoch 020 | Train F1: 0.9707 | Val F1: 0.9409 | Best: 0.9428
Epoch 025 | Train F1: 0.9738 | Val F1: 0.9462 | Best: 0.9531
Epoch 030 | Train F1: 0.9798 | Val F1: 0.9478 | Best: 0.9591
Epoch 035 | Train F1: 0.9851 | Val F1: 0.9643 | Best: 0.9643
Epoch 040 | Train F1: 0.9872 | Val F1: 0.9557 | Best: 0.9643
Epoch 045 | Train F1: 0.9877 | Val F1: 0.9704 | Best: 0.9751
Epoch 050 | Train F1: 0.9941 | Val F1: 0.9677 | Best: 0.9751
Epoch 055 | Train F1: 0.9973 | Val F1: 0.9642 | Best: 0.9764
Epoch 060 | Train F1: 0.9979 | Val F1: 0.9748 | Best: 0.9764
Epoch 065 | Train F1: 0.9995 | Val F1: 0.9733 | Best: 0.9764
Epoch 070 | Train F1: 0.9995 | Val F1: 0.9696 | Best: 0.9764
Early stopping at epoch 73
Finished | Best epoch: 53 | Best Val F1: 0.976391

Random_

/tmp/ipykernel_1577/3525986821.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer,num_layers=num_layers)


Epoch 001 | Train F1: 0.0855 | Val F1: 0.0490 | Best: 0.0490
Epoch 005 | Train F1: 0.8271 | Val F1: 0.8197 | Best: 0.8197
Epoch 010 | Train F1: 0.9524 | Val F1: 0.9605 | Best: 0.9700
Epoch 015 | Train F1: 0.9837 | Val F1: 0.9885 | Best: 0.9885
Epoch 020 | Train F1: 0.9912 | Val F1: 0.9885 | Best: 0.9885
Epoch 025 | Train F1: 0.9949 | Val F1: 0.9894 | Best: 0.9894
Epoch 030 | Train F1: 0.9949 | Val F1: 0.9894 | Best: 0.9947
Epoch 035 | Train F1: 0.9965 | Val F1: 0.9894 | Best: 0.9947
Epoch 040 | Train F1: 0.9971 | Val F1: 0.9894 | Best: 0.9947
Epoch 045 | Train F1: 0.9976 | Val F1: 0.9894 | Best: 0.9947
Early stopping at epoch 48
Finished | Best epoch: 28 | Best Val F1: 0.994706

Model seed: 123

Training: ['DA01', 'DA06', 'DA07', 'DA09', 'DA12'] | Seed: 123


...

Epoch 001 | Train F1: 0.1438 | Val F1: 0.1693 | Best: 0.1693
Epoch 005 | Train F1: 0.8057 | Val F1: 0.8136 | Best: 0.8136
Epoch 010 | Train F1: 0.9651 | Val F1: 0.8809 | Best: 0.8809
Epoch 015 | Train F1: 0.9845 | Val F1: 0.9372 | Best: 0.9372
Epoch 020 | Train F1: 0.9931 | Val F1: 0.9613 | Best: 0.9628
Epoch 025 | Train F1: 0.9957 | Val F1: 0.9677 | Best: 0.9810
Epoch 030 | Train F1: 0.9984 | Val F1: 0.9810 | Best: 0.9948
Epoch 035 | Train F1: 0.9995 | Val F1: 0.9874 | Best: 0.9948
Epoch 040 | Train F1: 1.0000 | Val F1: 0.9965 | Best: 0.9965
Epoch 045 | Train F1: 1.0000 | Val F1: 0.9928 | Best: 0.9965
Epoch 050 | Train F1: 1.0000 | Val F1: 0.9928 | Best: 0.9965
Epoch 055 | Train F1: 1.0000 | Val F1: 0.9912 | Best: 0.9965
Epoch 060 | Train F1: 1.0000 | Val F1: 0.9928 | Best: 0.9965
Early stopping at epoch 60
Finished | Best epoch: 40 | Best Val F1: 0.996479

Model seed: 123

Training: ['DA04', 'DA06', 'DA09', 'DA12', 'DA13', 'DA15'] | Seed: 123


/tmp/ipykernel_1577/3525986821.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer,num_layers=num_layers)


Epoch 001 | Train F1: 0.1190 | Val F1: 0.1290 | Best: 0.1290
Epoch 005 | Train F1: 0.8003 | Val F1: 0.6390 | Best: 0.7763
Epoch 010 | Train F1: 0.9500 | Val F1: 0.9050 | Best: 0.9050
Epoch 015 | Train F1: 0.9856 | Val F1: 0.9543 | Best: 0.9606
Epoch 020 | Train F1: 0.9920 | Val F1: 0.9794 | Best: 0.9912
Epoch 025 | Train F1: 0.9963 | Val F1: 1.0000 | Best: 1.0000
Epoch 030 | Train F1: 0.9984 | Val F1: 0.9984 | Best: 1.0000
Epoch 035 | Train F1: 0.9989 | Val F1: 0.9984 | Best: 1.0000
Epoch 040 | Train F1: 0.9995 | Val F1: 0.9984 | Best: 1.0000
Epoch 045 | Train F1: 0.9995 | Val F1: 0.9984 | Best: 1.0000
Early stopping at epoch 45
Finished | Best epoch: 25 | Best Val F1: 1.0

Model seed: 2026

Training: ['DA04', 'DA06', 'DA09', 'DA12', 'DA13', 'DA15'] | Seed: 2026


/tmp/ipykernel_1577/3525986821.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer,num_layers=num_layers)


Epoch 001 | Train F1: 0.1110 | Val F1: 0.1248 | Best: 0.1248
Epoch 005 | Train F1: 0.8576 | Val F1: 0.8324 | Best: 0.8324
Epoch 010 | Train F1: 0.9502 | Val F1: 0.9368 | Best: 0.9368
Epoch 015 | Train F1: 0.9845 | Val F1: 0.9673 | Best: 0.9761
Epoch 020 | Train F1: 0.9901 | Val F1: 0.9932 | Best: 0.9967
Epoch 025 | Train F1: 0.9965 | Val F1: 1.0000 | Best: 1.0000
Epoch 030 | Train F1: 0.9976 | Val F1: 0.9928 | Best: 1.0000
Epoch 035 | Train F1: 1.0000 | Val F1: 1.0000 | Best: 1.0000
Epoch 040 | Train F1: 1.0000 | Val F1: 1.0000 | Best: 1.0000
Early stopping at epoch 42
Finished | Best epoch: 22 | Best Val F1: 1.0

Random candidate validation completed.


In [ ]:
# ============================================================
# Random Search final summary
# ============================================================

random_final_df = pd.DataFrame(random_final_results)


random_summary = ( random_final_df.groupby(["candidate", "n_sensors"]
    ).agg(
        mean_f1=("val_macro_f1","mean"),
        std_f1=( "val_macro_f1", "std" ),
        min_f1=( "val_macro_f1", "min" ),
        max_f1=( "val_macro_f1", "max" )).reset_index())


print("\nValidation F1 by model seed:")

display(
    random_final_df.pivot(
        index="candidate",
        columns="model_seed",
        values="val_macro_f1"
    ))


print("\nSummary:")
display(random_summary)


Validation F1 by model seed:


model_seed,7,42,123,2026
candidate,,,,
Random_4,0.981584,0.985137,0.958617,0.976391
Random_5,0.994706,0.991049,0.993923,0.987781
Random_6,0.996479,0.996479,1.000000,1.000000



Summary:


,candidate,n_sensors,mean_f1,std_f1,min_f1,max_f1
0,Random_4,4,0.975432,0.011771,0.958617,0.985137
1,Random_5,5,0.991865,0.003144,0.987781,0.994706
2,Random_6,6,0.998239,0.002033,0.996479,1.000000


In [ ]:
random_final_df.to_csv(RANDOM_DIR / "results_by_seed.csv", index=False)

random_summary.to_csv(RANDOM_DIR / "summary.csv", index=False)

print("Random Search results saved.")

Random Search results saved.


In [ ]:
# ============================================================
# Candidate validation
# Preliminary shortlist
# ============================================================

# All candidates must have the complete set of model seeds
if not summary["n_seeds"].eq(len(MODEL_SEEDS)).all():

    raise ValueError(
        "Complete all model seeds before screening candidates."
    )


# Paired loss for each model seed
paired_drop = final_results.pivot(
    index="candidate",
    columns="model_seed",
    values="f1_drop"
).reindex(columns=MODEL_SEEDS)


print(
    "\nF1 loss relative to Full-15 "
    "(percentage points):"
)

print(
    (100 * paired_drop).to_string(
        float_format=lambda x: f"{x:.3f}"
    )
)


# Full-input models are references, not reduced candidates
subset_summary = summary[
    ~summary["candidate"].isin(
        ["Full_15", "Full_12"]
    )
].copy()


# Apply the existing mean-loss criterion
shortlist = subset_summary[
    subset_summary["mean_drop"]
    <= MAX_MEAN_F1_DROP + 1e-12
].sort_values(
    ["n_sensors", "mean_f1"],
    ascending=[True, False]
)


print(
    "\nCandidates passing the agreed mean-loss screen:"
)

if shortlist.empty:

    print(
        "No reduced candidate passed. "
        "Do not change the threshold automatically."
    )

else:

    print(
        shortlist[
            [
                "candidate",
                "n_sensors",
                "mean_f1",
                "std_f1",
                "min_f1",
                "mean_drop",
                "worst_drop"
            ]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )

    smallest_k = int(
        shortlist["n_sensors"].min()
    )

    smallest_candidates = shortlist[
        shortlist["n_sensors"] == smallest_k
    ]

    print(
        "\nSmallest sensor count passing "
        "the mean-loss screen:",
        smallest_k
    )

    print(
        "Candidates:",
        smallest_candidates["candidate"].tolist()
    )

    print(
        "This is a preliminary shortlist, "
        "not a final sensor decision."
    )


shortlist.to_csv(
    FINAL_DIR / "preliminary_shortlist.csv",
    index=False
)

paired_drop.to_csv(
    FINAL_DIR / "paired_f1_drop_by_seed.csv"
)


F1 loss relative to Full-15 (percentage points):
model_seed  42     7     123   2026
candidate                          
Floor_4    0.693  1.510 1.328 1.347
Floor_5    0.163  0.000 1.224 1.061
Full_12    0.000 -0.163 0.000 0.000
Full_15    0.000  0.000 0.000 0.000
WithBase_4 0.677  0.366 0.000 0.163
WithBase_5 0.163  0.857 1.224 4.914
WithBase_6 0.000  0.327 0.490 0.880

Candidates passing the agreed mean-loss screen:
 candidate  n_sensors  mean_f1   std_f1   min_f1  mean_drop  worst_drop
WithBase_4          4 0.996576 0.003141 0.993234   0.003015    0.006766
   Floor_5          5 0.993471 0.005691 0.987760   0.006120    0.012240
WithBase_6          6 0.995349 0.003604 0.991200   0.004243    0.008800

Smallest sensor count passing the mean-loss screen: 4
Candidates: ['WithBase_4']
This is a preliminary shortlist, not a final sensor decision.


In [ ]:
# ============================================================
# Candidate validation
# Check saved model weights
# ============================================================

weights_inventory = final_results[
    [
        "candidate",
        "model_seed",
        "sensors",
        "weights_path"
    ]
].copy()


weights_inventory["weights_path"] = (
    weights_inventory["weights_path"]
    .fillna("")
    .astype(str)
)


weights_inventory["weights_available"] = (
    weights_inventory["weights_path"].apply(
        lambda path:
            bool(path.strip())
            and Path(path).is_file()
    )
)


print("\nSaved model weights:")

print(
    weights_inventory[
        [
            "candidate",
            "model_seed",
            "weights_available"
        ]
    ].to_string(index=False)
)


# Full-15 is the reference for the planned importance ranking
missing_reference_weights = weights_inventory[
    (weights_inventory["candidate"] == "Full_15")
    & (~weights_inventory["weights_available"])
]


print(
    "\nFull-15 references without "
    "a valid recorded weights file:"
)

if missing_reference_weights.empty:

    print(
        "None. Weight files are present; "
        "their contents still need validation."
    )

else:

    print(
        missing_reference_weights[
            ["candidate", "model_seed"]
        ].to_string(index=False)
    )

    print(
        "No model is retrained by this check."
    )


weights_inventory.to_csv(
    FINAL_DIR / "weights_inventory.csv",
    index=False
)


Saved model weights:
 candidate  model_seed  weights_available
   Floor_4          42              False
   Floor_4           7              False
   Floor_4         123              False
   Floor_4        2026              False
WithBase_4          42              False
WithBase_4           7               True
WithBase_4         123               True
WithBase_4        2026               True
   Floor_5          42              False
   Floor_5           7               True
   Floor_5         123               True
   Floor_5        2026               True
WithBase_5          42              False
WithBase_5           7               True
WithBase_5         123               True
WithBase_5        2026               True
WithBase_6          42              False
WithBase_6           7              False
WithBase_6         123              False
WithBase_6        2026              False
   Full_15          42               True
   Full_15           7               True
   Full_15  